# Extraction Walkthrough — Synthetic Fixture

Drives the docling-graph `/extract-pass` endpoint with a hand-crafted 2-paragraph fixture, prints the **exact request** sent to the service, the **raw response** (including the library log, which captures the LLM's structured output), and rolls up entities + fields + relationships at the end.

Unlike `ingest_walkthrough.ipynb` this notebook does **not** touch Postgres / MinIO / ArcadeDB — it's a pure side-channel against the running docling-graph container. Safe to run while the production worker is processing other documents.

**Prerequisite:** docling-graph running on `localhost:8002` (the default in `docker-compose.yml`). Verify with `curl http://localhost:8002/health`.

Active bundle: `air_defense_v3`. Field-group passes used here:
1. `radar_identity` — extracts RADAR_SYSTEM identity (system_name, type, etc.)
2. `radar_power_rf` — extracts power/RF numeric parameters per system
3. `radar_antenna` — extracts antenna parameters
4. `system_links` — extracts inter-entity relationships


## §1 Configuration + fixture text

The fake fixture mimics analyst prose: two distinct radar systems described with explicit numeric parameters that the field-group schema knows how to bind.

In [1]:
import json
import time
import urllib.request
from pprint import pprint

# ─── Notebook configuration ─────────────────────────────────────────────
# Inside the Jupyter container we resolve services via docker DNS.
# From the host you'd use http://localhost:8002 / 8005 instead.
DOCLING_GRAPH_BASE = "http://docling-graph:8002"
DOCLING_GRAPH_URL = f"{DOCLING_GRAPH_BASE}/extract-pass"
API_BASE = "http://api:8000"  # Worker-API for /v1/documents/{id}/docling
BUNDLE_KEY = "air_defense_v3"

# ─── DOCUMENT_SOURCE toggle ─────────────────────────────────────────────
# "synthetic" — drive against the hand-crafted 2-paragraph fixture below.
#               Clean prose, deterministic, fast. Best for verifying the
#               extraction service basics work end-to-end.
# "real"      — fetch the docling_document.json artifact for a real corpus
#               document via the worker-API. Use this to debug why the
#               LLM extracts cleanly from synthetic prose but produces
#               empty/garbage on real docs (markdown noise hypothesis).
DOCUMENT_SOURCE = "real"   # "synthetic" | "real"

# When DOCUMENT_SOURCE == "real", which document to load. Accepts:
#   - a filename substring (case-insensitive), e.g. "EWIRDB" or "Fan_Song"
#   - a full filename, e.g. "EWIRDB_Production.pdf"
#   - a full UUID, e.g. "ed27c371-f7ee-4cc8-a554-e18966de369d"
# The full picker list is printed by §2 when DOCUMENT_SOURCE == "real",
# so you can copy-paste a filename instead of looking up the UUID.
REAL_DOCUMENT_NAME = "SA-2 Guideline _ Зенитный Ракетный Комплекс С-75 Двина_Десна_Волхов.pdf"

# ─── SELECTED_PASSES — pick which schema(s) to run ──────────────────────
# Comment out passes you don't want to run. Set to a single-element list
# to debug one schema in isolation. The rollup at §16 tolerates missing
# passes by treating their output as empty.
#
# All available passes:
SELECTED_PASSES = [
    "radar_identity",
    "radar_power_rf",
    "radar_antenna",
    "radar_timing",
    "radar_modulation",
    "missile_identity",
    "missile_kinematics",
    "missile_guidance",
    "missile_airframe",
    "missile_speed_timing",
    "missile_propulsion",
    "system_links",
]

# ─── FORCE_JSON_MODE flag ───────────────────────────────────────────────
# Toggle this to compare strict schema-grammar mode (False) vs loose JSON
# mode (True) on the same fixture. gemma4:31b reliably handles loose JSON
# but fails strict-grammar with unterminated-string parse errors on
# field-rich field-group schemas (e.g. radar_power_rf on parameter-heavy
# documents).
#
# This is a docling-graph SERVICE-LEVEL setting read at startup, so the
# notebook can't change it per-call. Set this flag, then run §1b below
# to get the exact host command that applies the flag and restarts
# docling-graph.
FORCE_JSON_MODE = True  # ← set True to test loose-JSON fallback

# ─── TEMPERATURE override ────────────────────────────────────────────────
# Per-request override for the extraction LLM call. Plumbed through to
# /extract-pass via the optional `temperature` field on ExtractPassRequest
# (added 2026-04-30). Notebook-only — production worker callers omit the
# field and inherit the service-default DOCLING_GRAPH_LLM_TEMPERATURE=0.1.
#
#   None  → use server default (0.1 in current production)
#   0.0   → fully deterministic (historically caused EMPTY JSON on
#           llama3.3:70b — see config_builder.py:84 comment block)
#   0.1   → production default
#   0.3   → exploration; helps gemma4 escape malformed-JSON local minima
#   0.7   → high variance; baseline for "is the model competent at all?"
TEMPERATURE: float | None = 1.0 # set to e.g. 0.3 for an A/B test

# ─── LLM_BATCH_TOKEN_SIZE override ───────────────────────────────────────
# Per-request override for chunk_batches_by_token_limit's max_batch_tokens.
# Plumbed through /extract-pass via the optional llm_batch_token_size field
# on ExtractPassRequest. Notebook-only — production inherits the service
# default DOCLING_GRAPH_LLM_BATCH_TOKEN_SIZE=1024.
#
#   None  → use server default (1024 in current production)
#   1024  → production default — conservative, 1-2 chunks per LLM call
#   2048  → 2x — historically rolled back as too aggressive for Ollama
#           (config_builder.py:42-44). Worth retesting at temp=0 to see
#           if cross-chunk binding gains overcome JSON-emission risk.
#   3072  → 3x — only if 2048 doesn't blow up the JSON-failure rate
#   4096  → 4x — most aggressive; long-context attention degradation
#           likely hurts narrow field-group extraction
LLM_BATCH_TOKEN_SIZE: int | None = 1024  # set to e.g. 2048 for A/B

FAKE_TEXT = (
    # ── Radar paragraph 1 ────────────────────────────────────────────────
    "The Patriot AN/MPQ-65 is a multi-function phased-array fire-control "
    "radar deployed by the U.S. Army with the Patriot air-defense system. "
    "It operates in C-band at a nominal carrier frequency of 5500 MHz, "
    "with a peak transmitter output of 750 kW and a 35 dBi peak antenna "
    "gain. The array provides a 1.5 degree azimuth beamwidth.\n"
    "\n"
    # ── Radar paragraph 2 ────────────────────────────────────────────────
    "By contrast, the AN/SPY-6(V)1 air and missile defense radar (AMDR) "
    "operates in S-band around 3300 MHz with a peak transmit power of "
    "approximately 1500 kW. Its active electronically scanned array "
    "delivers 42 dBi gain with a 0.9 degree beamwidth and is integrated "
    "into the Aegis Combat System on Flight III destroyers.\n"
    "\n"
    # ── Missile paragraph 1 ──────────────────────────────────────────────
    "The MIM-104F Patriot Advanced Capability-3 (PAC-3) is the missile "
    "interceptor paired with the AN/MPQ-65 fire-control radar and is "
    "operational with the U.S. Army. The PAC-3 has a body length of "
    "5.2 meters and a body diameter of 0.255 meters, with a total launch "
    "mass of 316 kg. It achieves a maximum intercept range of 35 km and "
    "engages targets up to 25 km altitude, with a minimum engagement "
    "altitude of 0.05 km. The missile uses active radar homing guidance "
    "and reaches Mach 5 (approximately 1700 m/s). Its single-stage solid "
    "rocket booster produces 100 kN of thrust over a 2.5-second burn.\n"
    "\n"
    # ── Missile paragraph 2 ──────────────────────────────────────────────
    "The RIM-174 Standard Missile 6 (SM-6) Block IA is the air and "
    "missile defense interceptor paired with the AN/SPY-6(V)1 AESA radar "
    "on Aegis-equipped destroyers. The SM-6 has a body length of 6.55 "
    "meters and a diameter of 0.34 meters with a total launch mass of "
    "1500 kg. Maximum intercept range exceeds 240 km. The Mark 72 "
    "booster delivers approximately 290 kN thrust over a 6-second burn, "
    "after which the dual-pulse Mark 104 sustainer provides extended "
    "cruise. The SM-6 employs semi-active radar homing with terminal "
    "active radar guidance and achieves Mach 3.5 (approximately 1190 "
    "m/s)."
)

print(f"DOCUMENT_SOURCE = {DOCUMENT_SOURCE!r}")
print(f"SELECTED_PASSES = {SELECTED_PASSES}")
print(f"FORCE_JSON_MODE = {FORCE_JSON_MODE}")


DOCUMENT_SOURCE = 'real'
SELECTED_PASSES = ['radar_identity', 'radar_power_rf', 'radar_antenna', 'radar_timing', 'radar_modulation', 'missile_identity', 'missile_kinematics', 'missile_guidance', 'missile_airframe', 'missile_speed_timing', 'missile_propulsion', 'system_links']
FORCE_JSON_MODE = True


## §1b Apply the `FORCE_JSON_MODE` flag

`DOCLING_GRAPH_FORCE_JSON_MODE` is a service-level env var read once at startup, so flipping the flag above requires restarting docling-graph with the new value. Run the cell below to get the exact host command.

After running the host command, wait ~10s for the service to come back healthy, then continue with §2.

In [2]:
target_value = "true" if FORCE_JSON_MODE else "false"

# Best-effort: probe the running docling-graph to see whether the requested
# value is already in effect. The /health endpoint doesn't expose env, so
# we just confirm the service is reachable and let the operator verify
# DOCLING_GRAPH_FORCE_JSON_MODE in the rebuilt container's env.
try:
    with urllib.request.urlopen(f"{DOCLING_GRAPH_BASE}/health", timeout=5) as r:
        h = json.loads(r.read())
    print(f"docling-graph reachable (schema_count={h.get('schema_count')}, "
          f"pipeline_version={h.get('pipeline_version')})")
except Exception as exc:
    print(f"docling-graph NOT reachable: {exc}")

print()
print("=" * 72)
print(f"Apply FORCE_JSON_MODE={FORCE_JSON_MODE} — run from your HOST shell:")
print("=" * 72)
print()
print(f"  DOCLING_GRAPH_FORCE_JSON_MODE={target_value} \\")
print(f"      docker compose --profile split up -d --no-build "
      f"--force-recreate docling-graph")
print()
print("Verify the service picked up the new value:")
print()
print(f"  docker exec eip-mmdpp-docling-graph-1 env | "
      f"grep DOCLING_GRAPH_FORCE_JSON_MODE")
print()
print(f"Expected output: DOCLING_GRAPH_FORCE_JSON_MODE={target_value}")

docling-graph reachable (schema_count=12, pipeline_version=1.5.0)

Apply FORCE_JSON_MODE=True — run from your HOST shell:

  DOCLING_GRAPH_FORCE_JSON_MODE=true \
      docker compose --profile split up -d --no-build --force-recreate docling-graph

Verify the service picked up the new value:

  docker exec eip-mmdpp-docling-graph-1 env | grep DOCLING_GRAPH_FORCE_JSON_MODE

Expected output: DOCLING_GRAPH_FORCE_JSON_MODE=true


## §2 Build the DoclingDocument JSON

This is the **exact payload shape** the production worker sends. The DoclingDocument is the canonical structured form of any input document — even our fake plain text gets wrapped in this envelope. The LLM never sees this raw JSON; docling-graph chunks it into ~512-token pieces and feeds those to the model. But this is the input from the worker's point of view.

In [3]:
def build_synthetic_docling_document(text: str, name: str = "synthetic-fixture") -> dict:
    """Minimal valid DoclingDocument with one text paragraph (synthetic mode).

    Mirrors what app/workers/pipeline.py:_build_extract_pass_request sends
    when a real PDF has been Docling-parsed — except for our fake input
    we synthesize one `texts[]` entry instead of the dozens-to-hundreds
    that come out of OCR.
    """
    return {
        "schema_name": "DoclingDocument",
        "version": "1.0.0",
        "name": name,
        "origin": {
            "mimetype": "text/plain",
            "binary_hash": 1,
            "filename": "smoke.txt",
        },
        "furniture": {"name": "_root_", "self_ref": "#/furniture", "children": []},
        "body": {
            "name": "_root_",
            "self_ref": "#/body",
            "children": [{"$ref": "#/texts/0"}],
        },
        "groups": [],
        "pictures": [],
        "tables": [],
        "key_value_items": [],
        "form_items": [],
        "pages": {},
        "texts": [{
            "self_ref": "#/texts/0",
            "parent": {"$ref": "#/body"},
            "label": "text",
            "prov": [],
            "orig": text,
            "text": text,
        }],
    }


def fetch_real_docling_document(document_id: str, api_base: str = API_BASE) -> dict:
    """Fetch the persisted DoclingDocument for a real corpus doc.

    Hits the worker-API at /v1/documents/{document_id}/docling and unwraps
    the `document_json` field, which holds the canonical DoclingDocument
    that `prepare_document` persisted to MinIO and `derive_ontology_graph`
    re-reads. The endpoint also returns `markdown`, `images`, `filename`,
    `document_id` alongside it — those are not part of the DoclingDocument
    schema and are dropped here.
    """
    url = f"{api_base}/v1/documents/{document_id}/docling"
    with urllib.request.urlopen(url, timeout=30) as r:
        envelope = json.loads(r.read())
    doc_json = envelope.get("document_json")
    if not isinstance(doc_json, dict) or doc_json.get("schema_name") != "DoclingDocument":
        raise ValueError(
            f"unexpected /docling response shape for {document_id}: "
            f"keys={list(envelope.keys())}; expected envelope with document_json"
        )
    return doc_json


# ── Helpers for the picker (filename → UUID resolver) ──────────────────
import re as _re_uuid


def list_available_documents(api_base: str = API_BASE) -> list[dict]:
    """Return [{document_id, filename, status, stage}, ...] across all sources.

    Used by §2 to print the picker list and by `_resolve_document_ref` to
    translate a filename substring to a UUID.
    """
    with urllib.request.urlopen(f"{api_base}/v1/sources", timeout=10) as r:
        sources = json.loads(r.read())
    results = []
    for s in sources:
        sid = s["id"]
        with urllib.request.urlopen(
            f"{api_base}/v1/sources/{sid}/documents", timeout=10
        ) as r:
            for d in json.loads(r.read()):
                results.append({
                    "document_id": d["id"],
                    "filename": d.get("filename") or "",
                    "source_id": sid,
                    "status": d.get("pipeline_status"),
                    "stage": d.get("pipeline_stage"),
                })
    return results


_UUID_RE = _re_uuid.compile(
    r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$",
    _re_uuid.IGNORECASE,
)


def _resolve_document_ref(ref: str, api_base: str = API_BASE) -> str:
    """Resolve `ref` to a document UUID. Accepts UUID, exact filename, or
    case-insensitive filename substring. Raises if zero or >1 matches.
    """
    if _UUID_RE.match(ref):
        return ref
    docs = list_available_documents(api_base)
    needle = ref.lower()
    matches = [d for d in docs if needle in d["filename"].lower()]
    if not matches:
        raise ValueError(
            f"No document matches {ref!r}. Available filenames:\n" +
            "\n".join(f"  - {d['filename']}" for d in docs)
        )
    if len(matches) > 1:
        raise ValueError(
            f"{len(matches)} documents match {ref!r} — narrow it down:\n" +
            "\n".join(f"  - {d['filename']}" for d in matches)
        )
    return matches[0]["document_id"]


# ── Dispatch on DOCUMENT_SOURCE ────────────────────────────────────────
if DOCUMENT_SOURCE == "synthetic":
    doc = build_synthetic_docling_document(FAKE_TEXT)
    print(f"Loaded SYNTHETIC fixture: {len(FAKE_TEXT)} chars")
elif DOCUMENT_SOURCE == "real":
    # Show the picker list first so the user knows what they can request.
    available = list_available_documents()
    print(f"=== Available documents ({len(available)}) ===")
    for d in available:
        st = (d["status"] or "?")[:11]
        print(f"  [{st:<11s}] {d['filename']:<70s}  {d['document_id']}")
    print()

    resolved_id = _resolve_document_ref(REAL_DOCUMENT_NAME)
    doc = fetch_real_docling_document(resolved_id)
    n_texts = len(doc.get("texts", []) or [])
    n_pictures = len(doc.get("pictures", []) or [])
    n_tables = len(doc.get("tables", []) or [])
    total_chars = sum(len(t.get("text", "") or "") for t in doc.get("texts", []) or [])
    matched = next((d for d in available if d["document_id"] == resolved_id), None)
    matched_name = matched["filename"] if matched else "?"
    print(f"Loaded REAL document: {matched_name}")
    print(f"  document_id: {resolved_id}")
    print(f"  texts={n_texts}, pictures={n_pictures}, tables={n_tables}")
    print(f"  total text chars across all elements: {total_chars}")
else:
    raise ValueError(f"Unknown DOCUMENT_SOURCE: {DOCUMENT_SOURCE!r}; "
                     f"use 'synthetic' or 'real'")

# Print a small preview so you can confirm what got loaded.
preview_texts = [(t.get("text") or "")[:200] for t in doc.get("texts", [])[:3]]
print()
print("--- First 3 text-element previews ---")
for i, p in enumerate(preview_texts):
    print(f"  texts[{i}]: {p!r}")


=== Available documents (21) ===
  [PARTIAL_COM] Engagement and Fire Control Radars (S-Band, X-band).pdf                 52f83fdb-3494-448c-86c3-fefb3392eae1
  [PARTIAL_COM] SA-2 Guideline _ Зенитный Ракетный Комплекс С-75 Двина_Десна_Волхов.pdf  ff024eac-2665-4306-9bb1-00eff20c6bab
  [COMPLETE   ] SA-2 Surface-to-Air Missile _ National Museum of the United States Air Force™ _ Display.pdf  ed27c371-f7ee-4cc8-a554-e18966de369d
  [COMPLETE   ] S-75 Dvina.pdf                                                          5a449470-3aee-4d3e-8e8c-d51ae7f5742c
  [PARTIAL_COM] S-75 Dvina _ Military Wiki _ Fandom.pdf                                 2b28f71a-e303-43ce-a469-99728b4ea62a
  [COMPLETE   ] chinese_research_paper.pdf                                              cee412f9-95ce-4146-af53-f892cf54a9a0
  [COMPLETE   ] SA-2_and_SR-71_17_Apr_2020.pdf                                          3d05ba69-be20-49bd-a665-85f9ab2dccdb
  [COMPLETE   ] EWIRDB_Production.pdf                                 

## §2b Inspect THE EXACT MARKDOWN sent to the LLM (post-sanitizer)

This is the diagnostic that confirms the "input representation" hypothesis. The docling-graph library does NOT send the LLM raw `text` strings — it renders the `DoclingDocument` to markdown via `format_batch_markdown()` and feeds that to the model.

**Pre-extraction sanitization runs first.** `docker/docling-graph/app/main.py:_sanitize_docling_document` walks every text element and **blanks** (does not remove — `$ref` chains break) any element matching:

1. **Rule 1 — ad-tracking domain.** Substring match on the element text against `adroll.com`, `adrta.com`, `doubleclick.net`, `googletagmanager.com`, `googletagservices.com`, `googleadservices.com`, `google-analytics.com`, `googlesyndication.com`, `facebook.com/tr`, `amazon-adsystem.com`, `adservice.google`, `scorecardresearch.com`.
2. **Rule 2 — pure-link line.** Every non-blank line is a markdown link `[text](url)` or bare URL with optional list marker. Catches sidebar nav, related-link columns, share-on-X rows.
3. **Rule 3 — encoded blob.** A whitespace-delimited token ≥ 64 chars is either base64 (with padding `+`/`/`/`=` or mixed case + digits) OR contains ≥ 6 `%XX` percent-encoded triplets. Catches ad payloads (`adroll_ad_payload=__HIA9QB...`), `data:image/...;base64,...` embeds, and the residue of tracker URLs that docling fragmented across line breaks.

`label='caption'` elements are preserved unconditionally so image-description prose survives.

The cell below applies this sanitizer locally (mirroring main.py) before chunking, so the markdown you see is byte-identical to what the LLM receives in its USER prompt. The `texts_dropped / texts_in` line at the top shows how aggressive the cruft filter was on this document.

For real PDFs that went through Docling+OCR, the markdown can also include things the sanitizer does NOT touch:
- Pipe-delimited table rows (`| col1 | col2 |`)
- Image-caption blocks (`![](image_n)\nCaption: ...`) — preserved by design
- Header/footer repetition across pages
- OCR artifacts (`Зенитный` mis-tokenized as `3eHnTHbiM`)
- Citation footnote anchors

If post-sanitizer markdown is still noticeably noisy, that points at a sanitizer-rule gap worth a TODO entry.


In [4]:
import re as _re
from docling_core.types.doc import DoclingDocument
from docling_graph.core.extractors.document_chunker import DocumentChunker
from docling_graph.core.extractors.contracts.delta.helpers import chunk_batches_by_token_limit
from docling_graph.core.extractors.contracts.delta.prompts import format_batch_markdown


# ── Local mirror of docker/docling-graph/app/main.py:_sanitize_docling_document
# Source of truth lives in main.py; keep these in sync. The mirror lets §2b
# render the *exact* markdown the LLM sees post-sanitizer, with no round-trip
# back to the docling-graph service.
_AD_TRACKING_DOMAINS = (
    "adroll.com", "adrta.com", "doubleclick.net", "googletagmanager.com",
    "googletagservices.com", "googleadservices.com", "google-analytics.com",
    "googlesyndication.com", "facebook.com/tr", "amazon-adsystem.com",
    "adservice.google", "scorecardresearch.com",
)
_PURE_LINK_LINE = _re.compile(
    r"^\s*[-*]?\s*"
    r"(?:\[[^\]]*\]\([^)]+\)|https?://\S+|<https?://\S+>)"
    r"\s*$",
    flags=_re.IGNORECASE | _re.MULTILINE,
)
_BASE64_TOKEN = _re.compile(r"[A-Za-z0-9+/_-]{64,}={0,2}")
_PERCENT_TRIPLET = _re.compile(r"%[0-9A-Fa-f]{2}")


def _contains_encoded_blob(text: str) -> bool:
    if not isinstance(text, str) or not text:
        return False
    for m in _BASE64_TOKEN.finditer(text):
        tok = m.group(0)
        has_padding = ("=" in tok) or ("+" in tok) or ("/" in tok)
        has_mixed = (
            any(c.isupper() for c in tok)
            and any(c.islower() for c in tok)
            and any(c.isdigit() for c in tok)
        )
        if has_padding or has_mixed:
            return True
    for tok in text.split():
        if len(tok) < 64:
            continue
        if len(_PERCENT_TRIPLET.findall(tok)) >= 6:
            return True
    return False


def _looks_like_nav_or_tracking(text: str) -> bool:
    if not isinstance(text, str) or not text.strip():
        return False
    lowered = text.lower()
    for dom in _AD_TRACKING_DOMAINS:
        if dom in lowered:
            return True
    nonblank = [ln for ln in text.splitlines() if ln.strip()]
    if not nonblank:
        return False
    if all(_PURE_LINK_LINE.match(ln) for ln in nonblank):
        return True
    if _contains_encoded_blob(text):
        return True
    return False


def _sanitize_docling_document(doc: dict) -> tuple[dict, dict]:
    """Mirror of main.py:_sanitize_docling_document. Returns (sanitized_doc, stats).

    Combines text + hyperlink for rule matching — docling stores markdown
    link URLs in a separate `hyperlink` annotation, and the chunker
    re-renders [text](hyperlink) from both fields. A tracker URL in the
    hyperlink would survive blanking of just text/orig, so we match on
    the combined string and clear hyperlink alongside.
    """
    texts_in = list(doc.get("texts") or [])
    stats = {"texts_in": len(texts_in), "texts_dropped": 0}
    new_texts: list = []
    blanked = 0
    for t in texts_in:
        if not isinstance(t, dict):
            new_texts.append(t)
            continue
        label = (t.get("label") or "").lower()
        if label == "caption":
            new_texts.append(t)
            continue
        text_str = t.get("text") or t.get("orig") or ""
        hyperlink = t.get("hyperlink") or ""
        # Render the way the chunker will: `[text](hyperlink)` for
        # hyperlinked items, bare text otherwise. Run rules against
        # the rendered form so Rule 2 (pure-link-line) catches nav
        # items where docling split text/URL into separate fields.
        if hyperlink:
            rendered = f"[{text_str}]({hyperlink})"
        else:
            rendered = text_str
        if _looks_like_nav_or_tracking(rendered):
            blanked_t = dict(t)
            blanked_t["text"] = ""
            blanked_t["orig"] = ""
            if "hyperlink" in blanked_t:
                blanked_t["hyperlink"] = None
            new_texts.append(blanked_t)
            blanked += 1
            continue
        new_texts.append(t)
    stats["texts_dropped"] = blanked
    if blanked == 0:
        return doc, stats
    new_doc = dict(doc)
    new_doc["texts"] = new_texts
    return new_doc, stats


def inspect_doc_markdown(document: dict, *, max_batches: int | None = None,
                         max_chars_per_batch: int | None = 4000,
                         apply_sanitizer: bool = True):
    """Show the exact markdown rendered for each chunk-batch sent to the LLM.

    No LLM call. Pure local rendering using the SAME library functions
    docling-graph uses internally — so the output is byte-identical to
    what the LLM sees in its USER prompt (modulo the prompt scaffolding
    around it).

    By default the same pre-extraction sanitizer that runs on the service
    side is applied here too, so the rendered markdown reflects the rules
    described in §2b. Pass apply_sanitizer=False to inspect the raw
    pre-sanitizer view (useful for diffing against the post-sanitizer one).

    Set max_chars_per_batch=None to dump entire batch contents (long).
    """
    if apply_sanitizer:
        sanitized_doc_json, sani_stats = _sanitize_docling_document(document)
    else:
        sanitized_doc_json, sani_stats = document, {"texts_in": len(document.get("texts") or []), "texts_dropped": 0}

    docling_doc = DoclingDocument.model_validate(sanitized_doc_json)
    chunker = DocumentChunker(
        tokenizer_name="sentence-transformers/all-MiniLM-L6-v2",
        chunk_max_tokens=512,
        merge_peers=True,
    )
    chunks = chunker.chunk_document(docling_doc)
    token_counts = [chunker.tokenizer.count_tokens(c) for c in chunks]
    batch_plan = chunk_batches_by_token_limit(chunks, token_counts, max_batch_tokens=1024)

    print(f"document: name={docling_doc.name!r}  sanitizer={'on' if apply_sanitizer else 'OFF'}")
    print(f"  texts_dropped/texts_in = {sani_stats['texts_dropped']}/{sani_stats['texts_in']}"
          f" ({100.0 * sani_stats['texts_dropped'] / sani_stats['texts_in']:.1f}% blanked)"
          if sani_stats['texts_in'] else
          f"  texts_dropped/texts_in = 0/0")
    print(f"  total_chunks={len(chunks)}  total_tokens={sum(token_counts)}  total_batches={len(batch_plan)}")
    print()

    n = max_batches if max_batches is not None else len(batch_plan)
    for batch_idx, selected_batch in enumerate(batch_plan[:n]):
        batch_texts = [text for _idx, text, _tok in selected_batch]
        batch_markdown = format_batch_markdown(batch_texts)
        batch_tokens = sum(tok for _i, _t, tok in selected_batch)
        print("=" * 78)
        print(f"BATCH {batch_idx} — {len(batch_texts)} chunks, {batch_tokens} tokens, "
              f"{len(batch_markdown)} chars")
        print("=" * 78)
        if max_chars_per_batch and len(batch_markdown) > max_chars_per_batch:
            print(batch_markdown[:max_chars_per_batch])
            print(f"\n... [+{len(batch_markdown) - max_chars_per_batch} more chars] ...")
        else:
            print(batch_markdown)
        print()
    if n < len(batch_plan):
        print(f"... [{len(batch_plan) - n} more batches not shown; "
              f"call with max_batches=None to see all]")


# Run it on whichever document is loaded — sanitizer ON by default so the
# output reflects the LLM's actual view. Pass apply_sanitizer=False to
# inspect the raw pre-sanitizer view for comparison.
inspect_doc_markdown(doc, max_batches=10)


/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

document: name='tmp2ak8gy9j'  sanitizer=on
  texts_dropped/texts_in = 65/308 (21.1% blanked)
  total_chunks=55  total_tokens=20077  total_batches=25

BATCH 0 — 3 chunks, 783 tokens, 3648 chars
--- CHUNK 1 ---
- 

--- CHUNK 2 ---

This image is classified as a photo (confidence: low), with engineering_drawing as a plausible alternate if the figure is interpreted as a simplified geometric symbol. The visual evidence consists of a low-resolution, blurred raster graphic depicting a light-grey or light-blue multi-pointed starburst centered on a dark-blue background. Physically, the object is a roughly symmetrical polygon with approximately eight points, though significant pixelation and blurring obscure precise vertex definitions and edge geometry. There is no visible OCR text, foreign-language script, or identifiable technical components, markings, or dimensions.

The image provides no technical characteristics, performance indicators, or functional data related to the S-75 SAM system desc

## §3 Helper — POST to /extract-pass and capture everything

This wraps the HTTP call. Returns the request body (so you can see exactly what was sent) and the full response. On the server side this triggers:

```
main.py:extract_pass(...)
  → load_bundle_manifest("air_defense_v3")          # find pass by name
  → load_pass_template(...)                          # Pydantic model
  → docling_graph.run_pipeline(..., llm_client=OllamaChatClient(...))
    → DocumentChunker.chunk(doc, max_tokens=512)
    → for each batch: OllamaChatClient.get_json_response(model=gemma4:31b, schema=template)
        → POST /v1/chat/completions to OllamaPool.acquire() URL
  → graph_to_pass_output(...)                        # node-link → pass_output
  → service_postprocess + identity_gate + evidence_gate
  → ExtractPassResponse
```

**Note (post-OllamaPool refactor):** docling-graph used to drive every LLM call through `LiteLLMClient`. Since the OllamaPool refactor (`app/services/ollama_pool_client.py` + the mirrored copy at `docker/docling-graph/app/ollama_pool_client.py`) it injects an `OllamaChatClient` directly into `PipelineConfig(llm_client=...)`, which talks to Ollama's `/v1/chat/completions` over the URL pool from `OLLAMA_LLM_BASE_URLS` (JSON-array env var; falls back to singular `OLLAMA_LLM_BASE_URL`, then `OLLAMA_BASE_URL`). Library logs in `diagnostics.library_log` now read `Initialized LlmBackend with client: OllamaChatClient` (was `LiteLLMClient`).

**Pool diagnostics:** when `DOCLING_GRAPH_DEBUG_ENDPOINTS=true`, `GET http://localhost:${DOCLING_GRAPH_PORT:-8002}/debug/routing-metrics` returns per-URL request counts so you can confirm fan-out across the pool. Returns 404 by default.

In [5]:
  import json
                                                                      
  for i, t in enumerate(doc.get("texts", [])):                      
      if not isinstance(t, dict):                                     
          continue                                                    
      text_str = t.get("text") or ""
      blob = json.dumps(t)                                            
      if "adroll" in blob.lower() or "doubleclick" in blob.lower():   
          print("--- texts[{}] ---".format(i))                        
          print("  text ({} chars): {!r}".format(len(text_str),       
  text_str[:120]))                                                    
          orig_str = t.get("orig") or ""                            
          print("  orig: {!r}".format(orig_str[:120]))                
          for k, v in t.items():                                    
              if k in ("text", "orig"):                               
                  continue                                            
              if isinstance(v, str):                                
                  preview = v[:120]                                   
              else:                                              
                  preview = json.dumps(v)[:200]                     
              print("  {}: {}".format(k, preview))                    
          print()
                      

--- texts[4] ---
  text (48 chars): 'Ready to win bigger; faster and smarter with AI?'
  orig: 'Ready to win bigger; faster and smarter with AI?'
  self_ref: #/texts/4
  parent: {"$ref": "#/groups/0"}
  children: []
  content_layer: body
  label: list_item
  prov: [{"page_no": 1, "bbox": {"l": 94.66666666666667, "t": 754.3333333333334, "r": 309.0, "b": 710.6666666666666, "coord_origin": "BOTTOMLEFT"}, "charspan": [0, 48]}]
  hyperlink: http://d.adroll.com/click/?adroll_insertion_id=48760b031b457241b2fc010a98a6d01c&adroll_pixalate_click_url=https%3A//adrt
  enumerated: false
  marker: 



In [6]:
def call_extract_pass(
    pass_name: str,
    document: dict,
    *,
    upstream_entities: list[dict] | None = None,
    timeout: int = 14400,
    temperature: float | None = None,
    llm_batch_token_size: int | None = None,
):
    """POST one /extract-pass call. Returns (request_body, response_dict, elapsed_seconds).

    `upstream_entities` is required for `document_plus_entity_refs` passes
    (e.g. system_links).

    `temperature` overrides the service-wide DOCLING_GRAPH_LLM_TEMPERATURE
    when not None. Falls back to the module-level TEMPERATURE flag from §1
    when no per-call value is supplied.
    """
    request_body = {
        "bundle_key": BUNDLE_KEY,
        "pass_name": pass_name,
        "document_id": f"notebook-{pass_name}",
        "docling_document_json": document,
    }
    if upstream_entities:
        request_body["upstream_entities"] = upstream_entities
    eff_temp = temperature if temperature is not None else TEMPERATURE
    if eff_temp is not None:
        request_body["temperature"] = float(eff_temp)
    eff_batch = llm_batch_token_size if llm_batch_token_size is not None else LLM_BATCH_TOKEN_SIZE
    if eff_batch is not None:
        request_body["llm_batch_token_size"] = int(eff_batch)
    payload = json.dumps(request_body).encode()
    req = urllib.request.Request(
        DOCLING_GRAPH_URL,
        data=payload,
        headers={"Content-Type": "application/json"},
    )
    t0 = time.monotonic()
    with urllib.request.urlopen(req, timeout=timeout) as r:
        response = json.loads(r.read())
    elapsed = time.monotonic() - t0
    _record_extraction_outcome(pass_name, response, elapsed, eff_temp, eff_batch)
    return request_body, response, elapsed


# ── Outcome tracker ────────────────────────────────────────────────────
# Every call_extract_pass(...) invocation auto-records into this dict so a
# final summary cell can report JSON-failure rate vs temperature without
# requiring per-pass instrumentation.
extraction_outcomes: list[dict] = []


def _record_extraction_outcome(pass_name: str, response: dict, elapsed: float,
                                temperature: float | None,
                                batch_tokens: int | None) -> None:
    diag = response.get("diagnostics") or {}
    log = diag.get("library_log") or ""
    md = response.get("metadata") or {}
    pipeline_error = bool(diag.get("pipeline_error"))
    no_valid_json = "No valid JSON returned" in log
    quality_gate_fail = "Quality gate failed" in log
    structured_sparse = "Warning: Structured output appears sparse" in log
    structured_failed = "Warning: Structured output failed" in log
    legacy_retry = "retrying legacy" in log or "retrying with legacy" in log
    nodes = md.get("node_count") or 0
    edges = md.get("edge_count") or 0
    batch_count = diag.get("batch_count")
    sanitize = diag.get("input_sanitize") or {}
    texts_in = sanitize.get("texts_in")
    texts_dropped = sanitize.get("texts_dropped")
    extraction_outcomes.append({
        "pass": pass_name,
        "temperature": temperature,
        "batch_tokens": batch_tokens,
        "batch_count": batch_count,
        "elapsed_s": round(elapsed, 1),
        "pipeline_error": pipeline_error,
        "no_valid_json": no_valid_json,
        "quality_gate_fail": quality_gate_fail,
        "structured_sparse_retry": structured_sparse,
        "structured_failed_retry": structured_failed,
        "legacy_fallback": legacy_retry,
        "node_count": nodes,
        "edge_count": edges,
        "texts_in": texts_in,
        "texts_dropped": texts_dropped,
        "json_failed": pipeline_error or no_valid_json,
        "zero_yield": nodes == 0 and edges == 0,
    })


def print_outcome_summary():
    """Print a pass-by-pass and aggregate JSON-failure summary."""
    if not extraction_outcomes:
        print("No extraction calls recorded yet.")
        return
    print(f"=== Extraction outcomes (n={len(extraction_outcomes)}) ===")
    print(f"{'pass':<25s} {'temp':<6s} {'btok':<6s} {'bcnt':<5s} {'elap':<6s} "
          f"{'nodes':<6s} {'edges':<6s} {'sanit':<8s} {'json_fail':<10s} {'qgate':<6s} "
          f"{'legacy':<7s} {'zero':<5s}")
    for o in extraction_outcomes:
        t = "def" if o["temperature"] is None else f'{o["temperature"]:.2f}'
        b = "def" if o["batch_tokens"] is None else str(o["batch_tokens"])
        bc = "?" if o["batch_count"] is None else str(o["batch_count"])
        if o["texts_in"] is None or o["texts_dropped"] is None:
            sanit_col = "?"
        else:
            sanit_col = f"{o['texts_dropped']}/{o['texts_in']}"
        print(f"{o['pass']:<25s} {t:<6s} {b:<6s} {bc:<5s} {o['elapsed_s']:<6.1f} "
              f"{o['node_count']:<6d} {o['edge_count']:<6d} {sanit_col:<8s} "
              f"{('Y' if o['json_failed'] else '.'):<10s} "
              f"{('Y' if o['quality_gate_fail'] else '.'):<6s} "
              f"{('Y' if o['legacy_fallback'] else '.'):<7s} "
              f"{('Y' if o['zero_yield'] else '.'):<5s}")
    n = len(extraction_outcomes)
    n_jf = sum(1 for o in extraction_outcomes if o["json_failed"])
    n_qg = sum(1 for o in extraction_outcomes if o["quality_gate_fail"])
    n_lg = sum(1 for o in extraction_outcomes if o["legacy_fallback"])
    n_zy = sum(1 for o in extraction_outcomes if o["zero_yield"])
    n_pe = sum(1 for o in extraction_outcomes if o["pipeline_error"])
    sani_in = sum((o["texts_in"] or 0) for o in extraction_outcomes)
    sani_dropped = sum((o["texts_dropped"] or 0) for o in extraction_outcomes)
    sani_pct = (100.0 * sani_dropped / sani_in) if sani_in else 0.0
    temps = sorted({o["temperature"] for o in extraction_outcomes}, key=lambda v: -1 if v is None else v)
    batches = sorted({o["batch_tokens"] for o in extraction_outcomes}, key=lambda v: -1 if v is None else v)
    temp_str = ", ".join("default" if t is None else f"{t:.2f}" for t in temps)
    batch_str = ", ".join("default" if b is None else str(b) for b in batches)
    print()
    print(f"Temperatures tested:  {temp_str}")
    print(f"Batch sizes tested:   {batch_str}")
    print(f"  json_failed (pipeline_error OR no_valid_json):  {n_jf:>3d}/{n} = {100*n_jf/n:5.1f}%")
    print(f"  pipeline_error (run_pipeline raised):           {n_pe:>3d}/{n} = {100*n_pe/n:5.1f}%")
    print(f"  quality_gate_failed (missing_root / empty):     {n_qg:>3d}/{n} = {100*n_qg/n:5.1f}%")
    print(f"  legacy_fallback engaged:                        {n_lg:>3d}/{n} = {100*n_lg/n:5.1f}%")
    print(f"  zero_yield (no entities + no edges):            {n_zy:>3d}/{n} = {100*n_zy/n:5.1f}%")
    print(f"  sanitize_dropped / texts_in (all passes):       {sani_dropped:>3d}/{sani_in:<3d} = {sani_pct:5.1f}%")
    cells = {}
    for o in extraction_outcomes:
        key = (o["temperature"], o["batch_tokens"])
        cells.setdefault(key, []).append(o)
    if len(cells) > 1:
        print()
        print(f"=== A/B cells ===")
        print(f"{'temp':<6s} {'batch':<8s} {'n':<3s} {'json_fail':<10s} {'qgate':<7s} {'legacy':<7s} {'zero':<5s} {'avg_elap':<10s}")
        for (t, b), outs in sorted(cells.items(), key=lambda kv: ((kv[0][0] or -1), (kv[0][1] or -1))):
            ts = "def" if t is None else f"{t:.2f}"
            bs = "def" if b is None else str(b)
            ns = len(outs)
            jf = sum(1 for o in outs if o["json_failed"])
            qg = sum(1 for o in outs if o["quality_gate_fail"])
            lg = sum(1 for o in outs if o["legacy_fallback"])
            zy = sum(1 for o in outs if o["zero_yield"])
            ae = sum(o["elapsed_s"] for o in outs) / ns
            print(f"{ts:<6s} {bs:<8s} {ns:<3d} {jf:>2d}/{ns:<7d} "
                  f"{qg:>2d}/{ns:<4d} {lg:>2d}/{ns:<4d} {zy:>2d}/{ns:<2d} {ae:>6.1f}s")


# Quick health check
with urllib.request.urlopen(f"{DOCLING_GRAPH_BASE}/health", timeout=5) as r:
    print(json.loads(r.read()))
print(f"TEMPERATURE override = {TEMPERATURE!r}")


{'status': 'ok', 'schema_count': 12, 'extraction_contract': 'delta', 'pipeline_version': '1.5.0'}
TEMPERATURE override = 1.0


## §3b Inspect what the LLM actually sees

The `/extract-pass` API call is the *outer* envelope. Inside docling-graph, the request gets fanned out into one or more LLM calls. Each LLM call gets:

1. A **SYSTEM prompt** — the global instructions (rules for emitting nodes, identity validation, FORBIDDEN-name policy, etc.). Source-of-truth: `ontology_bundles/_shared/prompt_rules.py::DELTA_SYSTEM_PROMPT` — overridden into the library prompt at request time.
2. A **USER prompt** — assembled per batch from:
   - The `path_catalog_block` (Pydantic-derived field catalog with descriptions)
   - The `schema_semantic_guide` (per-field guidance generated from the schema)
   - The `batch_markdown` (one or more chunks of the doc, format-rendered)
   - Optional `global_context` (first chunk preview) and `already_found` (prior-batch entities)
3. A **format= directive** to Ollama:
   - `format="json"` (loose) when `DOCLING_GRAPH_FORCE_JSON_MODE=true` OR the schema is over the 20 KB threshold
   - `format=<sanitized JSON Schema>` (strict grammar) otherwise — Ollama enforces JSON-Schema-conforming output token-by-token

The cell below defines `inspect_llm_prompt(pass_name)` — it reproduces docling-graph's internal assembly using the *same* library functions, without making an HTTP call. Each pass cell that follows calls it before the actual API call, so you can see exactly what's being asked of the LLM.

In [7]:
"""Helper that reconstructs the LLM prompt + schema for any pass.

Mirrors docling-graph's internal assembly path
(docker/docling-graph/app/main.py and the docling-graph library).
"""
from importlib import import_module
from docling_core.types.doc import DoclingDocument
from docling_graph.core.extractors.document_chunker import DocumentChunker
from docling_graph.core.extractors.contracts.delta.helpers import chunk_batches_by_token_limit
from docling_graph.core.extractors.contracts.delta.catalog import build_delta_node_catalog
from docling_graph.core.extractors.contracts.delta.schema_mapper import (
    build_catalog_prompt_block, build_delta_semantic_guide,
)
from docling_graph.core.extractors.contracts.delta.prompts import (
    get_delta_batch_prompt, format_batch_markdown,
)
# Service-side rewrite: the docling-graph service replaces the library's
# default system prompt with the source-of-truth in
# ontology_bundles/_shared/prompt_rules.py. Match that here.
from ontology_bundles._shared.prompt_rules import DELTA_SYSTEM_PROMPT

# All 12 active passes from manifest.yaml. Mirrors PASS_MODULES from the
# ingest_walkthrough notebook so this notebook can stand alone.
PASS_MODULES = {
    "radar_identity":       ("ontology_bundles.air_defense_v3.extraction_schemas.radar_identity",       "RadarIdentityPass"),
    "radar_power_rf":       ("ontology_bundles.air_defense_v3.extraction_schemas.radar_power_rf",       "RadarPowerRfPass"),
    "radar_antenna":        ("ontology_bundles.air_defense_v3.extraction_schemas.radar_antenna",        "RadarAntennaPass"),
    "radar_timing":         ("ontology_bundles.air_defense_v3.extraction_schemas.radar_timing",         "RadarTimingPass"),
    "radar_modulation":     ("ontology_bundles.air_defense_v3.extraction_schemas.radar_modulation",     "RadarModulationPass"),
    "missile_identity":     ("ontology_bundles.air_defense_v3.extraction_schemas.missile_identity",     "MissileIdentityPass"),
    "missile_kinematics":   ("ontology_bundles.air_defense_v3.extraction_schemas.missile_kinematics",   "MissileKinematicsPass"),
    "missile_guidance":     ("ontology_bundles.air_defense_v3.extraction_schemas.missile_guidance",     "MissileGuidancePass"),
    "missile_airframe":     ("ontology_bundles.air_defense_v3.extraction_schemas.missile_airframe",     "MissileAirframePass"),
    "missile_speed_timing": ("ontology_bundles.air_defense_v3.extraction_schemas.missile_speed_timing", "MissileSpeedTimingPass"),
    "missile_propulsion":   ("ontology_bundles.air_defense_v3.extraction_schemas.missile_propulsion",   "MissilePropulsionPass"),
    "system_links":         ("ontology_bundles.air_defense_v3.extraction_schemas.system_links",         "SystemLinksPass"),
}

# Cap big sections so output stays scannable; flip these to None to dump everything.
SCHEMA_PRINT_CAP = None
USER_PROMPT_CAP  = None


def inspect_llm_prompt(pass_name, document=None, *, batch_index=0):
    """Print the SYSTEM + USER prompts, JSON Schema, and format-mode for one pass."""
    if pass_name not in PASS_MODULES:
        raise ValueError(f"Unknown pass_name: {pass_name!r}; choices: {list(PASS_MODULES)}")
    if document is None:
        document = doc

    mod_path, cls_name = PASS_MODULES[pass_name]
    template_cls = getattr(import_module(mod_path), cls_name)

    # 1. Chunk + batch the doc the same way production does
    docling_doc = DoclingDocument.model_validate(document)
    chunker = DocumentChunker(
        tokenizer_name="sentence-transformers/all-MiniLM-L6-v2",
        chunk_max_tokens=512,
        merge_peers=True,
    )
    chunks = chunker.chunk_document(docling_doc)
    token_counts = [chunker.tokenizer.count_tokens(c) for c in chunks]
    batch_plan = chunk_batches_by_token_limit(chunks, token_counts, max_batch_tokens=1024)

    if batch_index >= len(batch_plan):
        print(f"batch_index={batch_index} out of range (only {len(batch_plan)} batches)")
        return

    selected_batch = batch_plan[batch_index]
    batch_texts = [text for _idx, text, _tok in selected_batch]
    batch_markdown = format_batch_markdown(batch_texts)

    # 2. Build catalog + semantic guide from the Pydantic template
    catalog        = build_delta_node_catalog(template_cls)
    catalog_block  = build_catalog_prompt_block(catalog)
    schema_dict    = template_cls.model_json_schema()
    semantic_guide = build_delta_semantic_guide(template_cls, schema_dict)

    first_chunk = chunks[0].strip() if chunks else ""
    global_context = (first_chunk[:600] + ("..." if len(first_chunk) > 600 else "")) if first_chunk else None

    # 3. Compose the prompt; override the library's system prompt with the
    #    service-side source-of-truth (matches docling-graph/app/main.py).
    _orig = get_delta_batch_prompt
    def _patched(**kw):
        r = _orig(**kw)
        if isinstance(r, dict) and "system" in r:
            r["system"] = DELTA_SYSTEM_PROMPT
        return r

    prompt = _patched(
        batch_markdown=batch_markdown,
        schema_semantic_guide=semantic_guide,
        path_catalog_block=catalog_block,
        batch_index=batch_index,
        total_batches=len(batch_plan),
        global_context=global_context,
        already_found=None,
    )

    # 4. Determine format mode the service will use for this call
    schema_str = json.dumps(schema_dict)
    threshold  = 20000  # docking-graph config_builder default
    if FORCE_JSON_MODE:
        format_mode = f"json (loose)  ← FORCE_JSON_MODE=true"
    elif len(schema_str) > threshold:
        format_mode = f"json (loose)  ← schema {len(schema_str)} chars > {threshold} threshold"
    else:
        format_mode = f"<schema> (strict grammar; schema {len(schema_str)} chars)"

    # 5. Render
    print(f"========== {pass_name}  template={cls_name} ==========")
    print(f"chunks={len(chunks)}  total_tokens={sum(token_counts)}  batches={len(batch_plan)}  batch_index={batch_index}")
    print(f"format_mode → {format_mode}")
    print(f"catalog_paths={len(catalog.nodes)}  semantic_guide_chars={len(semantic_guide)}")
    print()
    print("--- SYSTEM PROMPT ---")
    print(prompt["system"])
    print()
    print("--- USER PROMPT ---")
    user = prompt["user"]
    print(user[:USER_PROMPT_CAP] + ("\n[...truncated]" if USER_PROMPT_CAP and len(user) > USER_PROMPT_CAP else ""))
    print()
    print("--- JSON SCHEMA (template_cls.model_json_schema()) ---")
    sd = json.dumps(schema_dict, indent=2)
    print(sd[:SCHEMA_PRINT_CAP] + ("\n[...truncated]" if SCHEMA_PRINT_CAP and len(sd) > SCHEMA_PRINT_CAP else ""))


# Sanity check: list pass names
print("inspect_llm_prompt() ready. Pass names:")
for p in PASS_MODULES:
    print(f"  - {p}")

inspect_llm_prompt() ready. Pass names:
  - radar_identity
  - radar_power_rf
  - radar_antenna
  - radar_timing
  - radar_modulation
  - missile_identity
  - missile_kinematics
  - missile_guidance
  - missile_airframe
  - missile_speed_timing
  - missile_propulsion
  - system_links


## §3c Initialize default empty results for unrun passes

When `SELECTED_PASSES` excludes a pass, its variables (`req_<x>`, `resp_<x>`, `t_<x>`) won't be defined. The §16 rollup references all of them, so we initialize empties up-front. Cells whose pass is in `SELECTED_PASSES` will overwrite these defaults.

In [8]:
_EMPTY_RESPONSE = {"pass_output": {}, "diagnostics": {}}

# Defaults so §16 rollup tolerates skipped passes
req_id  = {}; resp_id  = _EMPTY_RESPONSE; t_id  = 0.0
req_pr  = {}; resp_pr  = _EMPTY_RESPONSE; t_pr  = 0.0
req_an  = {}; resp_an  = _EMPTY_RESPONSE; t_an  = 0.0
req_rt  = {}; resp_rt  = _EMPTY_RESPONSE; t_rt  = 0.0
req_rm  = {}; resp_rm  = _EMPTY_RESPONSE; t_rm  = 0.0
req_mi  = {}; resp_mi  = _EMPTY_RESPONSE; t_mi  = 0.0
req_mk  = {}; resp_mk  = _EMPTY_RESPONSE; t_mk  = 0.0
req_mg  = {}; resp_mg  = _EMPTY_RESPONSE; t_mg  = 0.0
req_ma  = {}; resp_ma  = _EMPTY_RESPONSE; t_ma  = 0.0
req_mst = {}; resp_mst = _EMPTY_RESPONSE; t_mst = 0.0
req_mp  = {}; resp_mp  = _EMPTY_RESPONSE; t_mp  = 0.0
req_sl  = {}; resp_sl  = _EMPTY_RESPONSE; t_sl  = 0.0

print(f"Defaults initialized. Will run only: {SELECTED_PASSES}")


Defaults initialized. Will run only: ['radar_identity', 'radar_power_rf', 'radar_antenna', 'radar_timing', 'radar_modulation', 'missile_identity', 'missile_kinematics', 'missile_guidance', 'missile_airframe', 'missile_speed_timing', 'missile_propulsion', 'system_links']


## §4 Pass 1 — `radar_identity`

Extracts the high-level identity for each radar mentioned in the text: `system_name`, `nomenclature`, `radar_type`, `band`, etc. — the *who* layer. Subsequent passes populate per-system numeric fields against these identities.

In [9]:
if 'radar_identity' in SELECTED_PASSES:
    # What the LLM is asked
    inspect_llm_prompt("radar_identity")
    print("\n" + "=" * 72 + "\n")

    # Actual API call
    req_id, resp_id, t_id = call_extract_pass("radar_identity", doc)

    print(f"-- elapsed: {t_id:.1f}s --\n")
    print(f"-- request: bundle={req_id.get('bundle_key')} pass={req_id.get('pass_name')} doc_chars={len(json.dumps(req_id.get('docling_document_json',{})))} temp={req_id.get('temperature','def')} batch={req_id.get('llm_batch_token_size','def')} --")
    print()
    print("-- response: pass_output.radar_systems --")
    for system in resp_id.get("pass_output", {}).get("radar_systems", []):
        print(json.dumps(system, indent=2))
else:
    print('[skip radar_identity] not in SELECTED_PASSES')


[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

========== radar_identity  template=RadarIdentityPass ==========
chunks=60  total_tokens=23023  batches=28  batch_index=0
format_mode → json (loose)  ← FORCE_JSON_MODE=true
catalog_paths=2  semantic_guide_chars=2321

--- SYSTEM PROMPT ---
You are a high-precision graph extraction engine for **radar and missile-domain graph construction**. Return **ONLY valid JSON** with exactly two top-level keys: "nodes" and "relationships".

## Output Contract

Return exactly:

```json
{"nodes": [...], "relationships": [...]}
```

Each node must have this shape:

```json
{
  "path": "<catalog path>",
  "node_type": "<optional>",
  "ids": {...},
  "parent": {"path": "<catalog path>", "ids": {...}} or null,
  "properties": {...}
}
```

No markdown. No explanations. No comments. No prose outside the JSON.

---

## Core Extraction Objective

Extract **only entities and relationships directly evidenced in the current batch document content**.

Use the **Template Path Catalog** and **Semantic Field Guidanc

In [10]:
if 'radar_identity' in SELECTED_PASSES:
    # Service-level diagnostics — what docling-graph saw internally
    diags = resp_id.get("diagnostics", {})
    print("chunk_count:", diags.get("chunk_count"))
    print("batch_count:", diags.get("batch_count"))
    print("parallel_workers:", diags.get("parallel_workers"))
    print("llm_batch_token_size:", diags.get("llm_batch_token_size"))
    print("path_counts:", diags.get("path_counts"))
    print("quality_gate.ok:", diags.get("quality_gate", {}).get("ok"))
    print("quality_gate.reasons:", diags.get("quality_gate", {}).get("reasons"))
    print()
    print("-- batch_timings (per-LLM-call elapsed) --")
    for bt in diags.get("batch_timings", []):
        print(f"  batch {bt.get('batch_index')}: {bt.get('elapsed_seconds'):.2f}s")
else:
    print('[skip radar_identity] not in SELECTED_PASSES')


chunk_count: 55
batch_count: 25
parallel_workers: 4
llm_batch_token_size: 1024
path_counts: {'': 1, 'radar_systems[]': 19}
quality_gate.ok: True
quality_gate.reasons: []

-- batch_timings (per-LLM-call elapsed) --
  batch 0: 32.85s
  batch 1: 331.65s
  batch 2: 54.14s
  batch 3: 22.60s
  batch 4: 25.69s
  batch 5: 36.91s
  batch 6: 296.82s
  batch 7: 23.44s
  batch 8: 20.77s
  batch 9: 331.60s
  batch 10: 63.95s
  batch 11: 74.49s
  batch 12: 29.95s
  batch 13: 33.56s
  batch 14: 16.19s
  batch 15: 23.38s
  batch 16: 66.80s
  batch 17: 71.98s
  batch 18: 312.31s
  batch 19: 104.28s
  batch 20: 28.90s
  batch 21: 43.69s
  batch 22: 37.31s
  batch 23: 112.56s
  batch 24: 18.52s


In [11]:
if 'radar_identity' in SELECTED_PASSES:
    # library_log captures the docling-graph library's stdout during this call —
    # this is the closest thing to seeing the LLM's raw JSON output (post-parse).
    # If you set DOCLING_GRAPH_FORCE_JSON_MODE=true and restart the service,
    # the warning lines about 'Structured output failed' / 'retrying with legacy
    # prompt-schema mode' should disappear here.
    print(diags.get("library_log", ""))
else:
    print('[skip radar_identity] not in SELECTED_PASSES')


[LlmBackend] Initialized with:
  • Client: OllamaChatClient
  • Model: gemma4:31b
[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True
[DocumentProcessor] Initialized with Classic OCR pipeline (English, French)
[ExtractorFactory] Created ManyToOneStrategy
[Extraction] PATCH GATE: contract='delta' has_chunk_batches=True has_chunker=True => can_chunk_batch=True
[Extraction] Using CHUNKED-BATCHES path (matches documented delta flow).
[DocumentProcessor] Extracted 55 chunks with metadata
[DeltaExtraction] Running delta extraction (55 chunks)...
[DeltaExtraction] Calling LLM (batch mode)...
[LlmBackend] Successfully extracted data from DoclingDocument
[GraphConverter] Pre-registering models for deterministic node IDs...
[GraphConverter] Running automatic graph cleanup...
[GraphCleaner] Starting cleanup: 25 nodes, 24 edges
[GraphCleaner] Cleanup complete:
  • Removed 0 phantom nodes
  • Merged 0 duplicate no

## §5 Pass 2 — `radar_power_rf`

Per-system numeric fields: `nominal_rf_mhz`, `tx_peak_power_kw`, `prf_hz`, `pulse_width_us`, etc. The LLM has to bind the same `system_name` it used in `radar_identity` (free-form prompt — there is no formal `upstream_entities` for `document_only` field-group passes; identity is pinned by the prompt template referencing `system_name` as the primary key).

In [12]:
if 'radar_power_rf' in SELECTED_PASSES:
    # What the LLM is asked
    inspect_llm_prompt("radar_power_rf")
    print("\n" + "=" * 72 + "\n")

    # Actual API call
    req_pr, resp_pr, t_pr = call_extract_pass("radar_power_rf", doc)

    print(f"-- elapsed: {t_pr:.1f}s --\n")
    print(f"-- request: pass={req_pr.get('pass_name')} doc_chars={len(json.dumps(req_pr.get('docling_document_json',{})))} --")
    print()
    print("-- response: pass_output.radar_systems --")
    for system in resp_pr.get("pass_output", {}).get("radar_systems", []):
        populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
        print(
            f"{system.get('system_name'):<25} "
            f"populated_fields={len(populated)}: {sorted(populated.keys())}"
        )
        print(json.dumps(populated, indent=2))
        print()
else:
    print('[skip radar_power_rf] not in SELECTED_PASSES')


[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

========== radar_power_rf  template=RadarPowerRfPass ==========
chunks=60  total_tokens=23023  batches=28  batch_index=0
format_mode → json (loose)  ← FORCE_JSON_MODE=true
catalog_paths=2  semantic_guide_chars=940

--- SYSTEM PROMPT ---
You are a high-precision graph extraction engine for **radar and missile-domain graph construction**. Return **ONLY valid JSON** with exactly two top-level keys: "nodes" and "relationships".

## Output Contract

Return exactly:

```json
{"nodes": [...], "relationships": [...]}
```

Each node must have this shape:

```json
{
  "path": "<catalog path>",
  "node_type": "<optional>",
  "ids": {...},
  "parent": {"path": "<catalog path>", "ids": {...}} or null,
  "properties": {...}
}
```

No markdown. No explanations. No comments. No prose outside the JSON.

---

## Core Extraction Objective

Extract **only entities and relationships directly evidenced in the current batch document content**.

Use the **Template Path Catalog** and **Semantic Field Guidance*

## §6 Pass 3 — `radar_antenna`

Antenna parameters: `gain_dbi`, `azimuth_beamwidth_deg`, `elevation_beamwidth_deg`, `aperture_m`, scan-type enum, etc.

In [13]:
if 'radar_antenna' in SELECTED_PASSES:
    # What the LLM is asked
    inspect_llm_prompt("radar_antenna")
    print("\n" + "=" * 72 + "\n")

    # Actual API call
    req_an, resp_an, t_an = call_extract_pass("radar_antenna", doc)

    print(f"-- elapsed: {t_an:.1f}s --\n")
    print(f"-- request: pass={req_an.get('pass_name')} doc_chars={len(json.dumps(req_an.get('docling_document_json',{})))} --")
    print()
    print("-- response: pass_output.radar_systems --")
    for system in resp_an.get("pass_output", {}).get("radar_systems", []):
        populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
        print(json.dumps(populated, indent=2))
        print()
else:
    print('[skip radar_antenna] not in SELECTED_PASSES')


[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

========== radar_antenna  template=RadarAntennaPass ==========
chunks=60  total_tokens=23023  batches=28  batch_index=0
format_mode → json (loose)  ← FORCE_JSON_MODE=true
catalog_paths=2  semantic_guide_chars=1498

--- SYSTEM PROMPT ---
You are a high-precision graph extraction engine for **radar and missile-domain graph construction**. Return **ONLY valid JSON** with exactly two top-level keys: "nodes" and "relationships".

## Output Contract

Return exactly:

```json
{"nodes": [...], "relationships": [...]}
```

Each node must have this shape:

```json
{
  "path": "<catalog path>",
  "node_type": "<optional>",
  "ids": {...},
  "parent": {"path": "<catalog path>", "ids": {...}} or null,
  "properties": {...}
}
```

No markdown. No explanations. No comments. No prose outside the JSON.

---

## Core Extraction Objective

Extract **only entities and relationships directly evidenced in the current batch document content**.

Use the **Template Path Catalog** and **Semantic Field Guidance*

## §7 `radar_timing`

Field-group pass — populates `radar_systems[]` with the per-system fields owned by this group. See `ontology_bundles/air_defense_v3/extraction_schemas/radar_timing.py` for the field list.

In [14]:
if 'radar_timing' in SELECTED_PASSES:
    # What the LLM is asked
    inspect_llm_prompt("radar_timing")
    print("\n" + "=" * 72 + "\n")

    # Actual API call
    req_rt, resp_rt, t_rt = call_extract_pass("radar_timing", doc)

    print(f"-- elapsed: {t_rt:.1f}s --\n")
    print(f"-- request: pass={req_rt.get('pass_name')} doc_chars={len(json.dumps(req_rt.get('docling_document_json',{})))} --")
    print()
    print("-- response: pass_output.radar_systems --")
    for system in resp_rt.get("pass_output", {}).get("radar_systems", []) or []:
        populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
        print(json.dumps(populated, indent=2))
        print()

else:
    print('[skip radar_timing] not in SELECTED_PASSES')


[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

========== radar_timing  template=RadarTimingPass ==========
chunks=60  total_tokens=23023  batches=28  batch_index=0
format_mode → json (loose)  ← FORCE_JSON_MODE=true
catalog_paths=2  semantic_guide_chars=1132

--- SYSTEM PROMPT ---
You are a high-precision graph extraction engine for **radar and missile-domain graph construction**. Return **ONLY valid JSON** with exactly two top-level keys: "nodes" and "relationships".

## Output Contract

Return exactly:

```json
{"nodes": [...], "relationships": [...]}
```

Each node must have this shape:

```json
{
  "path": "<catalog path>",
  "node_type": "<optional>",
  "ids": {...},
  "parent": {"path": "<catalog path>", "ids": {...}} or null,
  "properties": {...}
}
```

No markdown. No explanations. No comments. No prose outside the JSON.

---

## Core Extraction Objective

Extract **only entities and relationships directly evidenced in the current batch document content**.

Use the **Template Path Catalog** and **Semantic Field Guidance** 

## §8 `radar_modulation`

Field-group pass — populates `radar_systems[]` with the per-system fields owned by this group. See `ontology_bundles/air_defense_v3/extraction_schemas/radar_modulation.py` for the field list.

In [15]:
if 'radar_modulation' in SELECTED_PASSES:
    # What the LLM is asked
    inspect_llm_prompt("radar_modulation")
    print("\n" + "=" * 72 + "\n")

    # Actual API call
    req_rm, resp_rm, t_rm = call_extract_pass("radar_modulation", doc)

    print(f"-- elapsed: {t_rm:.1f}s --\n")
    print(f"-- request: pass={req_rm.get('pass_name')} doc_chars={len(json.dumps(req_rm.get('docling_document_json',{})))} --")
    print()
    print("-- response: pass_output.radar_systems --")
    for system in resp_rm.get("pass_output", {}).get("radar_systems", []) or []:
        populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
        print(json.dumps(populated, indent=2))
        print()

else:
    print('[skip radar_modulation] not in SELECTED_PASSES')


[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

========== radar_modulation  template=RadarModulationPass ==========
chunks=60  total_tokens=23023  batches=28  batch_index=0
format_mode → json (loose)  ← FORCE_JSON_MODE=true
catalog_paths=2  semantic_guide_chars=1435

--- SYSTEM PROMPT ---
You are a high-precision graph extraction engine for **radar and missile-domain graph construction**. Return **ONLY valid JSON** with exactly two top-level keys: "nodes" and "relationships".

## Output Contract

Return exactly:

```json
{"nodes": [...], "relationships": [...]}
```

Each node must have this shape:

```json
{
  "path": "<catalog path>",
  "node_type": "<optional>",
  "ids": {...},
  "parent": {"path": "<catalog path>", "ids": {...}} or null,
  "properties": {...}
}
```

No markdown. No explanations. No comments. No prose outside the JSON.

---

## Core Extraction Objective

Extract **only entities and relationships directly evidenced in the current batch document content**.

Use the **Template Path Catalog** and **Semantic Field Gui

## §9 `missile_identity`

Field-group pass — populates `missile_systems[]` with the per-system fields owned by this group. See `ontology_bundles/air_defense_v3/extraction_schemas/missile_identity.py` for the field list.

In [16]:
if 'missile_identity' in SELECTED_PASSES:
    # What the LLM is asked
    inspect_llm_prompt("missile_identity")
    print("\n" + "=" * 72 + "\n")

    # Actual API call
    req_mi, resp_mi, t_mi = call_extract_pass("missile_identity", doc)

    print(f"-- elapsed: {t_mi:.1f}s --\n")
    print(f"-- request: pass={req_mi.get('pass_name')} doc_chars={len(json.dumps(req_mi.get('docling_document_json',{})))} --")
    print()
    print("-- response: pass_output.missile_systems --")
    for system in resp_mi.get("pass_output", {}).get("missile_systems", []) or []:
        populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
        print(json.dumps(populated, indent=2))
        print()

else:
    print('[skip missile_identity] not in SELECTED_PASSES')


[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

========== missile_identity  template=MissileIdentityPass ==========
chunks=60  total_tokens=23023  batches=28  batch_index=0
format_mode → json (loose)  ← FORCE_JSON_MODE=true
catalog_paths=2  semantic_guide_chars=1974

--- SYSTEM PROMPT ---
You are a high-precision graph extraction engine for **radar and missile-domain graph construction**. Return **ONLY valid JSON** with exactly two top-level keys: "nodes" and "relationships".

## Output Contract

Return exactly:

```json
{"nodes": [...], "relationships": [...]}
```

Each node must have this shape:

```json
{
  "path": "<catalog path>",
  "node_type": "<optional>",
  "ids": {...},
  "parent": {"path": "<catalog path>", "ids": {...}} or null,
  "properties": {...}
}
```

No markdown. No explanations. No comments. No prose outside the JSON.

---

## Core Extraction Objective

Extract **only entities and relationships directly evidenced in the current batch document content**.

Use the **Template Path Catalog** and **Semantic Field Gui

## §10 `missile_kinematics`

Field-group pass — populates `missile_systems[]` with the per-system fields owned by this group. See `ontology_bundles/air_defense_v3/extraction_schemas/missile_kinematics.py` for the field list.

In [17]:
if 'missile_kinematics' in SELECTED_PASSES:
    # What the LLM is asked
    inspect_llm_prompt("missile_kinematics")
    print("\n" + "=" * 72 + "\n")

    # Actual API call
    req_mk, resp_mk, t_mk = call_extract_pass("missile_kinematics", doc)

    print(f"-- elapsed: {t_mk:.1f}s --\n")
    print(f"-- request: pass={req_mk.get('pass_name')} doc_chars={len(json.dumps(req_mk.get('docling_document_json',{})))} --")
    print()
    print("-- response: pass_output.missile_systems --")
    for system in resp_mk.get("pass_output", {}).get("missile_systems", []) or []:
        populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
        print(json.dumps(populated, indent=2))
        print()

else:
    print('[skip missile_kinematics] not in SELECTED_PASSES')


[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

========== missile_kinematics  template=MissileKinematicsPass ==========
chunks=60  total_tokens=23023  batches=28  batch_index=0
format_mode → json (loose)  ← FORCE_JSON_MODE=true
catalog_paths=2  semantic_guide_chars=1237

--- SYSTEM PROMPT ---
You are a high-precision graph extraction engine for **radar and missile-domain graph construction**. Return **ONLY valid JSON** with exactly two top-level keys: "nodes" and "relationships".

## Output Contract

Return exactly:

```json
{"nodes": [...], "relationships": [...]}
```

Each node must have this shape:

```json
{
  "path": "<catalog path>",
  "node_type": "<optional>",
  "ids": {...},
  "parent": {"path": "<catalog path>", "ids": {...}} or null,
  "properties": {...}
}
```

No markdown. No explanations. No comments. No prose outside the JSON.

---

## Core Extraction Objective

Extract **only entities and relationships directly evidenced in the current batch document content**.

Use the **Template Path Catalog** and **Semantic Field

## §11 `missile_guidance`

Field-group pass — populates `missile_systems[]` with the per-system fields owned by this group. See `ontology_bundles/air_defense_v3/extraction_schemas/missile_guidance.py` for the field list.

In [18]:
if 'missile_guidance' in SELECTED_PASSES:
    # What the LLM is asked
    inspect_llm_prompt("missile_guidance")
    print("\n" + "=" * 72 + "\n")

    # Actual API call
    req_mg, resp_mg, t_mg = call_extract_pass("missile_guidance", doc)

    print(f"-- elapsed: {t_mg:.1f}s --\n")
    print(f"-- request: pass={req_mg.get('pass_name')} doc_chars={len(json.dumps(req_mg.get('docling_document_json',{})))} --")
    print()
    print("-- response: pass_output.missile_systems --")
    for system in resp_mg.get("pass_output", {}).get("missile_systems", []) or []:
        populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
        print(json.dumps(populated, indent=2))
        print()

else:
    print('[skip missile_guidance] not in SELECTED_PASSES')


[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

========== missile_guidance  template=MissileGuidancePass ==========
chunks=60  total_tokens=23023  batches=28  batch_index=0
format_mode → json (loose)  ← FORCE_JSON_MODE=true
catalog_paths=2  semantic_guide_chars=1166

--- SYSTEM PROMPT ---
You are a high-precision graph extraction engine for **radar and missile-domain graph construction**. Return **ONLY valid JSON** with exactly two top-level keys: "nodes" and "relationships".

## Output Contract

Return exactly:

```json
{"nodes": [...], "relationships": [...]}
```

Each node must have this shape:

```json
{
  "path": "<catalog path>",
  "node_type": "<optional>",
  "ids": {...},
  "parent": {"path": "<catalog path>", "ids": {...}} or null,
  "properties": {...}
}
```

No markdown. No explanations. No comments. No prose outside the JSON.

---

## Core Extraction Objective

Extract **only entities and relationships directly evidenced in the current batch document content**.

Use the **Template Path Catalog** and **Semantic Field Gui

## §12 `missile_airframe`

Field-group pass — populates `missile_systems[]` with the per-system fields owned by this group. See `ontology_bundles/air_defense_v3/extraction_schemas/missile_airframe.py` for the field list.

In [19]:
if 'missile_airframe' in SELECTED_PASSES:
    # What the LLM is asked
    inspect_llm_prompt("missile_airframe")
    print("\n" + "=" * 72 + "\n")

    # Actual API call
    req_ma, resp_ma, t_ma = call_extract_pass("missile_airframe", doc)

    print(f"-- elapsed: {t_ma:.1f}s --\n")
    print(f"-- request: pass={req_ma.get('pass_name')} doc_chars={len(json.dumps(req_ma.get('docling_document_json',{})))} --")
    print()
    print("-- response: pass_output.missile_systems --")
    for system in resp_ma.get("pass_output", {}).get("missile_systems", []) or []:
        populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
        print(json.dumps(populated, indent=2))
        print()

else:
    print('[skip missile_airframe] not in SELECTED_PASSES')


[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

========== missile_airframe  template=MissileAirframePass ==========
chunks=60  total_tokens=23023  batches=28  batch_index=0
format_mode → json (loose)  ← FORCE_JSON_MODE=true
catalog_paths=2  semantic_guide_chars=955

--- SYSTEM PROMPT ---
You are a high-precision graph extraction engine for **radar and missile-domain graph construction**. Return **ONLY valid JSON** with exactly two top-level keys: "nodes" and "relationships".

## Output Contract

Return exactly:

```json
{"nodes": [...], "relationships": [...]}
```

Each node must have this shape:

```json
{
  "path": "<catalog path>",
  "node_type": "<optional>",
  "ids": {...},
  "parent": {"path": "<catalog path>", "ids": {...}} or null,
  "properties": {...}
}
```

No markdown. No explanations. No comments. No prose outside the JSON.

---

## Core Extraction Objective

Extract **only entities and relationships directly evidenced in the current batch document content**.

Use the **Template Path Catalog** and **Semantic Field Guid

## §13 `missile_speed_timing`

Field-group pass — populates `missile_systems[]` with the per-system fields owned by this group. See `ontology_bundles/air_defense_v3/extraction_schemas/missile_speed_timing.py` for the field list.

In [20]:
if 'missile_speed_timing' in SELECTED_PASSES:
    # What the LLM is asked
    inspect_llm_prompt("missile_speed_timing")
    print("\n" + "=" * 72 + "\n")

    # Actual API call
    req_mst, resp_mst, t_mst = call_extract_pass("missile_speed_timing", doc)

    print(f"-- elapsed: {t_mst:.1f}s --\n")
    print(f"-- request: pass={req_mst.get('pass_name')} doc_chars={len(json.dumps(req_mst.get('docling_document_json',{})))} --")
    print()
    print("-- response: pass_output.missile_systems --")
    for system in resp_mst.get("pass_output", {}).get("missile_systems", []) or []:
        populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
        print(json.dumps(populated, indent=2))
        print()

else:
    print('[skip missile_speed_timing] not in SELECTED_PASSES')


[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

========== missile_speed_timing  template=MissileSpeedTimingPass ==========
chunks=60  total_tokens=23023  batches=28  batch_index=0
format_mode → json (loose)  ← FORCE_JSON_MODE=true
catalog_paths=2  semantic_guide_chars=1580

--- SYSTEM PROMPT ---
You are a high-precision graph extraction engine for **radar and missile-domain graph construction**. Return **ONLY valid JSON** with exactly two top-level keys: "nodes" and "relationships".

## Output Contract

Return exactly:

```json
{"nodes": [...], "relationships": [...]}
```

Each node must have this shape:

```json
{
  "path": "<catalog path>",
  "node_type": "<optional>",
  "ids": {...},
  "parent": {"path": "<catalog path>", "ids": {...}} or null,
  "properties": {...}
}
```

No markdown. No explanations. No comments. No prose outside the JSON.

---

## Core Extraction Objective

Extract **only entities and relationships directly evidenced in the current batch document content**.

Use the **Template Path Catalog** and **Semantic Fi

## §14 `missile_propulsion`

Field-group pass — populates `missile_systems[]` with the per-system fields owned by this group. See `ontology_bundles/air_defense_v3/extraction_schemas/missile_propulsion.py` for the field list.

In [21]:
if 'missile_propulsion' in SELECTED_PASSES:
    # What the LLM is asked
    inspect_llm_prompt("missile_propulsion")
    print("\n" + "=" * 72 + "\n")

    # Actual API call
    req_mp, resp_mp, t_mp = call_extract_pass("missile_propulsion", doc)

    print(f"-- elapsed: {t_mp:.1f}s --\n")
    print(f"-- request: pass={req_mp.get('pass_name')} doc_chars={len(json.dumps(req_mp.get('docling_document_json',{})))} --")
    print()
    print("-- response: pass_output.missile_systems --")
    for system in resp_mp.get("pass_output", {}).get("missile_systems", []) or []:
        populated = {k: v for k, v in system.items() if v not in (None, "", [], {})}
        print(json.dumps(populated, indent=2))
        print()

else:
    print('[skip missile_propulsion] not in SELECTED_PASSES')


[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

========== missile_propulsion  template=MissilePropulsionPass ==========
chunks=60  total_tokens=23023  batches=28  batch_index=0
format_mode → json (loose)  ← FORCE_JSON_MODE=true
catalog_paths=2  semantic_guide_chars=2027

--- SYSTEM PROMPT ---
You are a high-precision graph extraction engine for **radar and missile-domain graph construction**. Return **ONLY valid JSON** with exactly two top-level keys: "nodes" and "relationships".

## Output Contract

Return exactly:

```json
{"nodes": [...], "relationships": [...]}
```

Each node must have this shape:

```json
{
  "path": "<catalog path>",
  "node_type": "<optional>",
  "ids": {...},
  "parent": {"path": "<catalog path>", "ids": {...}} or null,
  "properties": {...}
}
```

No markdown. No explanations. No comments. No prose outside the JSON.

---

## Core Extraction Objective

Extract **only entities and relationships directly evidenced in the current batch document content**.

Use the **Template Path Catalog** and **Semantic Field

## §15 `system_links` (relationships)

Cross-domain relationship pass. Now we DO have upstream entities (from `radar_identity` + `missile_identity`), so we pass them to the service so it can emit relationship edges (`ASSOCIATED_WITH` / `CUES`) between the radars and missiles in the prose.

In [22]:
if 'system_links' in SELECTED_PASSES:
    # Build upstream_entities from radar_identity + missile_identity outputs.
    # Mirrors what app/workers/pipeline.py:_run_single_pass does in production.
    upstream_entities = []
    for entity in resp_id.get("pass_output", {}).get("radar_systems", []) or []:
        name = entity.get("system_name")
        if name:
            upstream_entities.append({
                "ref_id":          f"RADAR_SYSTEM:{name}",
                "entity_type":     "RADAR_SYSTEM",
                "identity_values": {"system_name": name},
                "display_label":   name,
            })
    for entity in resp_mi.get("pass_output", {}).get("missile_systems", []) or []:
        name = entity.get("system_name")
        if name:
            upstream_entities.append({
                "ref_id":          f"MISSILE_SYSTEM:{name}",
                "entity_type":     "MISSILE_SYSTEM",
                "identity_values": {"system_name": name},
                "display_label":   name,
            })

    print(f"-- {len(upstream_entities)} upstream_entities pinned --")
    for u in upstream_entities:
        print(f"  {u['entity_type']:<14} {u['system_name'] if 'system_name' in u else u['display_label']}")
    print()

    # What the LLM is asked
    inspect_llm_prompt("system_links")
    print("\n" + "=" * 72 + "\n")

    # Actual API call (now with upstream_entities)
    try:
        req_sl, resp_sl, t_sl = call_extract_pass(
            "system_links", doc, upstream_entities=upstream_entities,
        )
        print(f"-- elapsed: {t_sl:.1f}s --\n")
        print(f"-- request: pass={req_sl.get('pass_name')} doc_chars={len(json.dumps(req_sl.get('docling_document_json',{})))} --")
        print()
        print("-- response: pass_output --")
        print(json.dumps(resp_sl.get("pass_output", {}), indent=2))
    except urllib.error.HTTPError as exc:
        print(f"HTTP {exc.code}: {exc.read().decode()[:500]}")
        resp_sl = {"pass_output": {}}

else:
    print('[skip system_links] not in SELECTED_PASSES')


-- 58 upstream_entities pinned --
  RADAR_SYSTEM   Fan Song
  RADAR_SYSTEM   Spoon Rest
  RADAR_SYSTEM   Fan Song E
  RADAR_SYSTEM   Side Net
  RADAR_SYSTEM   Amazonka
  RADAR_SYSTEM   RSNA-75
  RADAR_SYSTEM   RSN-75
  RADAR_SYSTEM   Parol
  RADAR_SYSTEM   Flat Face
  RADAR_SYSTEM   Squat Eye
  RADAR_SYSTEM   SNR-75
  RADAR_SYSTEM   SA-75 Desna
  RADAR_SYSTEM   PRV-10 Konus
  RADAR_SYSTEM   PRV-11 Vershina
  RADAR_SYSTEM   RD-75 Amazonka
  RADAR_SYSTEM   Fan Song A
  RADAR_SYSTEM   Spoon Rest D/E
  RADAR_SYSTEM   SNR-75M3
  RADAR_SYSTEM   SNR-75M3 Fan Song
  MISSILE_SYSTEM S-75
  MISSILE_SYSTEM HQ-2
  MISSILE_SYSTEM CSA-1
  MISSILE_SYSTEM 1D
  MISSILE_SYSTEM 13D
  MISSILE_SYSTEM DM
  MISSILE_SYSTEM DA
  MISSILE_SYSTEM DAM
  MISSILE_SYSTEM 20D
  MISSILE_SYSTEM DP
  MISSILE_SYSTEM DSU
  MISSILE_SYSTEM 5Ya23
  MISSILE_SYSTEM 15D
  MISSILE_SYSTEM V-750
  MISSILE_SYSTEM SA-10
  MISSILE_SYSTEM SA-2
  MISSILE_SYSTEM HQ-1
  MISSILE_SYSTEM 11D
  MISSILE_SYSTEM S-25
  MISSILE_SYSTEM SA-1
  MISSI

[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

========== system_links  template=SystemLinksPass ==========
chunks=60  total_tokens=23023  batches=28  batch_index=0
format_mode → json (loose)  ← FORCE_JSON_MODE=true
catalog_paths=2  semantic_guide_chars=1689

--- SYSTEM PROMPT ---
You are a high-precision graph extraction engine for **radar and missile-domain graph construction**. Return **ONLY valid JSON** with exactly two top-level keys: "nodes" and "relationships".

## Output Contract

Return exactly:

```json
{"nodes": [...], "relationships": [...]}
```

Each node must have this shape:

```json
{
  "path": "<catalog path>",
  "node_type": "<optional>",
  "ids": {...},
  "parent": {"path": "<catalog path>", "ids": {...}} or null,
  "properties": {...}
}
```

No markdown. No explanations. No comments. No prose outside the JSON.

---

## Core Extraction Objective

Extract **only entities and relationships directly evidenced in the current batch document content**.

Use the **Template Path Catalog** and **Semantic Field Guidance** 

## §16 Rollup — entities × fields, edges

Combine outputs from all 11 entity-emitting passes. Production merges via `extraction_merge.merge_and_resolve()`; here we just zip pass outputs by `system_name` and split by entity type.

In [23]:
"""§16 — production-parity rollup.

Mirrors what app/services/extraction_merge.py:merge_and_resolve does:
  1. Phase 0 — canonicalize_cross_pass_identities: detect entities split
     across passes (e.g. missile_identity emits 'MIM-104F', missile_airframe
     emits 'PAC-3' from the same prose) and rewrite to a single canonical
     system_name BEFORE merging.
  2. Phase 1 — merge: zip by canonical system_name; union all populated fields.

This makes the notebook rollup match what would land in the production graph
after merge_and_resolve runs.
"""
import re
from collections import defaultdict


def _identity_token_bag(entity: dict) -> set[str]:
    """Tokenize system_name + nomenclature + name into UPPERCASE token set,
    dropping single-char tokens."""
    parts = [entity.get(f) for f in ("system_name", "nomenclature", "name")
             if isinstance(entity.get(f), str)]
    text = " ".join(p for p in parts if p)
    return {t.upper() for t in re.split(r"[^A-Za-z0-9]+", text) if len(t) >= 2}


def _name_only_tokens(entity: dict) -> set[str]:
    name = entity.get("system_name") or ""
    return {t.upper() for t in re.split(r"[^A-Za-z0-9]+", name) if len(t) >= 2}


def _likely_same_entity(a: dict, b: dict) -> bool:
    a_name, b_name = a.get("system_name"), b.get("system_name")
    if not a_name or not b_name:
        return False
    if a_name == b_name:
        return True
    a_tokens = _name_only_tokens(a)
    b_tokens = _name_only_tokens(b)
    if not a_tokens or not b_tokens:
        return False
    return a_tokens.issubset(_identity_token_bag(b)) or \
           b_tokens.issubset(_identity_token_bag(a))


def _pick_canonical(names):
    cands = sorted({n for n in names if isinstance(n, str) and n}, key=lambda n: (len(n), n))
    return cands[0] if cands else ""


def canonicalize_systems(per_pass_lists):
    flat = [e for sub in per_pass_lists for e in sub]
    if len(flat) <= 1:
        return
    n = len(flat)
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(x, y):
        rx, ry = find(x), find(y)
        if rx != ry:
            parent[rx] = ry
    for i in range(n):
        for j in range(i + 1, n):
            if _likely_same_entity(flat[i], flat[j]):
                union(i, j)
    components = defaultdict(list)
    for i in range(n):
        components[find(i)].append(i)
    rewrites = 0
    for member_idxs in components.values():
        if len(member_idxs) <= 1:
            continue
        canonical = _pick_canonical([flat[i].get("system_name") for i in member_idxs])
        if not canonical:
            continue
        for i in member_idxs:
            if flat[i].get("system_name") != canonical:
                flat[i]["system_name"] = canonical
                rewrites += 1
    if rewrites:
        print(f"canonicalize_systems: rewrote {rewrites} system_name(s) "
              f"to collapse cross-pass duplicates")


# ── Gather per-pass entity lists ────────────────────────────────────────
# Defensive: pull each response from globals via .get() so a fresh kernel,
# a partial SELECTED_PASSES run, or a kernel restart doesn't crash the rollup
# with NameError. Missing passes contribute empty entity lists instead.
_EMPTY = {"pass_output": {}}
RADAR_PASSES = [
    ("identity",   globals().get("resp_id",  _EMPTY)),
    ("power_rf",   globals().get("resp_pr",  _EMPTY)),
    ("antenna",    globals().get("resp_an",  _EMPTY)),
    ("timing",     globals().get("resp_rt",  _EMPTY)),
    ("modulation", globals().get("resp_rm",  _EMPTY)),
]
MISSILE_PASSES = [
    ("identity",     globals().get("resp_mi",  _EMPTY)),
    ("kinematics",   globals().get("resp_mk",  _EMPTY)),
    ("guidance",     globals().get("resp_mg",  _EMPTY)),
    ("airframe",     globals().get("resp_ma",  _EMPTY)),
    ("speed_timing", globals().get("resp_mst", _EMPTY)),
    ("propulsion",   globals().get("resp_mp",  _EMPTY)),
]

radar_lists   = [resp.get("pass_output", {}).get("radar_systems",   []) or []
                 for _, resp in RADAR_PASSES]
missile_lists = [resp.get("pass_output", {}).get("missile_systems", []) or []
                 for _, resp in MISSILE_PASSES]

# ── Phase 0 — cross-pass canonicalization (mirrors production) ──────────
canonicalize_systems(radar_lists)
canonicalize_systems(missile_lists)

# ── Phase 1 — merge by canonical system_name ────────────────────────────
merged_radar   = defaultdict(dict)
merged_missile = defaultdict(dict)

for (label, _), entities in zip(RADAR_PASSES, radar_lists):
    for system in entities:
        name = system.get("system_name")
        if not name:
            continue
        for k, v in system.items():
            if v not in (None, "", [], {}):
                merged_radar[name][k] = v
        merged_radar[name].setdefault("_passes_seen_in", set()).add(label)

for (label, _), entities in zip(MISSILE_PASSES, missile_lists):
    for system in entities:
        name = system.get("system_name")
        if not name:
            continue
        for k, v in system.items():
            if v not in (None, "", [], {}):
                merged_missile[name][k] = v
        merged_missile[name].setdefault("_passes_seen_in", set()).add(label)


def _print_block(label, merged):
    print(f"\n=========== {label} ===========")
    for name, fields in merged.items():
        passes = sorted(fields.pop("_passes_seen_in", set()))
        print(f"\n--- {name} (seen in: {', '.join(passes)}) ---")
        for k in sorted(fields.keys()):
            print(f"  {k:<28} = {fields[k]}")


_print_block("RADAR_SYSTEMS", merged_radar)
_print_block("MISSILE_SYSTEMS", merged_missile)


canonicalize_systems: rewrote 97 system_name(s) to collapse cross-pass duplicates
canonicalize_systems: rewrote 117 system_name(s) to collapse cross-pass duplicates

=========== RADAR_SYSTEMS ===========

--- SNR-75 (seen in: antenna, identity, modulation, power_rf, timing) ---
  emitter_function             = FIRE_CONTROL
  nomenclature                 = SNR-75
  scan_type                    = CIRCULAR
  system_name                  = SNR-75

--- P-12 (seen in: antenna, identity, modulation, power_rf, timing) ---
  emitter_function             = FIRE_CONTROL
  nomenclature                 = P-18-2/P-18M
  system_name                  = P-12

--- Side Net (seen in: antenna, identity, modulation, power_rf, timing) ---
  emitter_function             = SEARCH
  nomenclature                 = PRV-10 Konus / PRV-11 Vershina
  system_name                  = Side Net

--- Amazonka (seen in: antenna, identity, modulation, power_rf, timing) ---
  emitter_function             = SEARCH
  nomencla

In [24]:
# Cleaner tabular view (one DataFrame per entity type)
_mr = globals().get("merged_radar")
_mm = globals().get("merged_missile")
if _mr is None or _mm is None:
    print("Run §16 (rollup) first — merged_radar / merged_missile not defined yet.")
else:
    try:
        import pandas as pd
        df_radar = pd.DataFrame.from_dict(_mr, orient="index").fillna("-")
        df_missile = pd.DataFrame.from_dict(_mm, orient="index").fillna("-")
        print("=== RADAR_SYSTEMS ===")
        print(df_radar)
        print()
        print("=== MISSILE_SYSTEMS ===")
        print(df_missile)
    except ImportError:
        print("pandas not installed; install with `pip install pandas`")


=== RADAR_SYSTEMS ===
             system_name                    nomenclature emitter_function  \
SNR-75            SNR-75                          SNR-75     FIRE_CONTROL   
P-12                P-12                    P-18-2/P-18M     FIRE_CONTROL   
Side Net        Side Net  PRV-10 Konus / PRV-11 Vershina           SEARCH   
Amazonka        Amazonka                           RD-75           SEARCH   
RSN-75            RSN-75                          RSN-75                -   
Parol              Parol                               -           SEARCH   
Flat Face      Flat Face                         P-15/19           SEARCH   
Squat Eye      Squat Eye                           P-15M           SEARCH   
SA-75 Desna  SA-75 Desna                               -                -   
RSN- 75M        RSN- 75M                               -                -   
RSN- 75V1      RSN- 75V1                               -                -   
RSN- 75V        RSN- 75V                              

In [25]:
# Relationships emitted across all passes (system_links is the canonical source).
# Defensive: missing pass responses (kernel restart, partial SELECTED_PASSES)
# contribute zero edges instead of crashing.
_EMPTY_RESP = {"pass_output": {}}
_resps = [globals().get(name, _EMPTY_RESP) for name in (
    "resp_id", "resp_pr", "resp_an", "resp_rt", "resp_rm",
    "resp_mi", "resp_mk", "resp_mg", "resp_ma", "resp_mst", "resp_mp",
    "resp_sl",
)]
edges = []
for resp in _resps:
    for edge in resp.get("pass_output", {}).get("relationships", []) or []:
        edges.append(edge)

if edges:
    print(f"-- {len(edges)} relationship(s) extracted --")
    for e in edges:
        print(json.dumps(e, indent=2))
else:
    print("No relationships extracted — system_links produced no edges (or system_links not in SELECTED_PASSES).")


-- 20 relationship(s) extracted --
{
  "rel_type": "CUES",
  "from_ref_id": "RADAR_SYSTEM:Spoon Rest",
  "to_ref_id": "RADAR_SYSTEM:Fan Song",
  "confidence": 0.9
}
{
  "rel_type": "CUES",
  "from_ref_id": "RADAR_SYSTEM:Flat Face",
  "to_ref_id": "RADAR_SYSTEM:Fan Song",
  "confidence": 0.9
}
{
  "rel_type": "CUES",
  "from_ref_id": "RADAR_SYSTEM:Squat Eye",
  "to_ref_id": "RADAR_SYSTEM:Fan Song",
  "confidence": 0.9
}
{
  "rel_type": "CUES",
  "from_ref_id": "RADAR_SYSTEM:Side Net",
  "to_ref_id": "RADAR_SYSTEM:Fan Song",
  "confidence": 0.9
}
{
  "rel_type": "CUES",
  "from_ref_id": "RADAR_SYSTEM:Amazonka",
  "to_ref_id": "RADAR_SYSTEM:Fan Song",
  "confidence": 0.9
}
{
  "rel_type": "ASSOCIATED_WITH",
  "from_ref_id": "RADAR_SYSTEM:Parol",
  "to_ref_id": "RADAR_SYSTEM:Fan Song",
  "confidence": 0.9
}
{
  "rel_type": "ASSOCIATED_WITH",
  "from_ref_id": "RADAR_SYSTEM:Spoon Rest",
  "to_ref_id": "RADAR_SYSTEM:Fan Song",
  "confidence": 0.9
}
{
  "rel_type": "ASSOCIATED_WITH",
  "from_r

In [26]:
# Final summary — defensive against missing pass variables
def _t(name):
    v = globals().get(name)
    return round(v, 1) if isinstance(v, (int, float)) else None

_mr = globals().get("merged_radar", {})
_mm = globals().get("merged_missile", {})
_edges = globals().get("edges", [])
summary = {
    "radar_systems_extracted":        list(_mr.keys()),
    "missile_systems_extracted":      list(_mm.keys()),
    "total_populated_radar_fields":   sum(len(f) for f in _mr.values()),
    "total_populated_missile_fields": sum(len(f) for f in _mm.values()),
    "per_pass_elapsed_seconds": {
        "radar_identity":       _t("t_id"),
        "radar_power_rf":       _t("t_pr"),
        "radar_antenna":        _t("t_an"),
        "radar_timing":         _t("t_rt"),
        "radar_modulation":     _t("t_rm"),
        "missile_identity":     _t("t_mi"),
        "missile_kinematics":   _t("t_mk"),
        "missile_guidance":     _t("t_mg"),
        "missile_airframe":     _t("t_ma"),
        "missile_speed_timing": _t("t_mst"),
        "missile_propulsion":   _t("t_mp"),
        "system_links":         _t("t_sl"),
    },
    "relationships": len(_edges),
}
print(json.dumps(summary, indent=2))


{
  "radar_systems_extracted": [
    "SNR-75",
    "P-12",
    "Side Net",
    "Amazonka",
    "RSN-75",
    "Parol",
    "Flat Face",
    "Squat Eye",
    "SA-75 Desna",
    "RSN- 75M",
    "RSN- 75V1",
    "RSN- 75V",
    "RSN- 75M4"
  ],
  "missile_systems_extracted": [
    "1D",
    "HQ-1",
    "CSA-1",
    "13D",
    "DM",
    "DA",
    "DAM",
    "20D",
    "DP",
    "DSU",
    "15D",
    "V-750",
    "11D",
    "Wasserfall",
    "S-75M1",
    "S-75M2",
    "S-75M4",
    "13DM",
    "13DA",
    "13DAM",
    "20DP",
    "20DSU",
    "AIM-9",
    "HQ2",
    "SM-90",
    "SM-90M",
    "SM-63"
  ],
  "total_populated_radar_fields": 28,
  "total_populated_missile_fields": 81,
  "per_pass_elapsed_seconds": {
    "radar_identity": 1576.0,
    "radar_power_rf": 991.5,
    "radar_antenna": 1113.4,
    "radar_timing": 835.8,
    "radar_modulation": 784.2,
    "missile_identity": 1741.3,
    "missile_kinematics": 1300.3,
    "missile_guidance": 1532.1,
    "missile_airframe": 1167.1,
    "m

## §17 Outcome summary — JSON-failure rate per temperature

`call_extract_pass()` auto-records every call into `extraction_outcomes`. This cell prints a per-pass table and aggregate failure rates so you can A/B different `TEMPERATURE` values against the same document and see which one minimizes `json_failed` / `quality_gate_failed` / `legacy_fallback`.

To run an A/B sweep, restart the kernel between runs (each kernel start clears `extraction_outcomes`), or call `extraction_outcomes.clear()` manually before re-running with a different `TEMPERATURE`.

In [27]:
if 'print_outcome_summary' in dir():
    print_outcome_summary()
else:
    print("Run §3 (call-helper) first — print_outcome_summary not defined yet.")


=== Extraction outcomes (n=12) ===
pass                      temp   btok   bcnt  elap   nodes  edges  sanit    json_fail  qgate  legacy  zero 
radar_identity            1.00   1024   25    1576.0 20     19     65/308   .          .      .       .    
radar_power_rf            1.00   1024   25    991.5  28     27     65/308   .          .      .       .    
radar_antenna             1.00   1024   25    1113.4 28     27     65/308   .          .      .       .    
radar_timing              1.00   1024   25    835.8  23     22     65/308   .          .      .       .    
radar_modulation          1.00   1024   25    784.2  30     29     65/308   .          .      .       .    
missile_identity          1.00   1024   25    1741.3 40     39     65/308   .          .      .       .    
missile_kinematics        1.00   1024   25    1300.3 48     47     65/308   .          .      .       .    
missile_guidance          1.00   1024   25    1532.1 45     44     65/308   .          .      .      

## §18 Inspect failed-batch traces

When a pass triggers any failure indicator (`pipeline_error` set, OR any of the library warnings — Quality gate failed / No valid JSON / Structured output failed / etc.), docling-graph now embeds the library's per-batch trace files in `response.diagnostics.failed_batch_traces`.

Each batch trace contains:
- `prompt.system` — the full system prompt
- `prompt.user` — the EXACT markdown the LLM saw (catalog block + semantic guide + batch chunks)
- `output.nodes` and `output.relationships` — what the LLM actually returned

Use this to debug a structured-output failure WITHOUT re-running the pass: scroll to the failing batch, inspect the user prompt content vs. the empty/sparse output, and identify which input pattern caused the model to fail.

In [28]:
def show_failed_batches(response: dict, *, max_user_chars: int = 4000,
                       max_output_nodes: int = 10):
    """Render the failed-batch traces from a single /extract-pass response.

    response: any of resp_id, resp_pr, resp_an, ... resp_sl after a pass call.
    """
    diag = (response or {}).get("diagnostics") or {}
    traces = diag.get("failed_batch_traces") or {}
    if not traces:
        print("(no failed_batch_traces — pass had no failure indicators)")
        return
    print(f"=== {len(traces)} failed batch trace(s) ===")
    for fname in sorted(traces.keys()):
        bt = traces[fname]
        if not isinstance(bt, dict):
            print(f"\n--- {fname}: <unparseable> ---")
            continue
        if "_load_error" in bt:
            print(f"\n--- {fname}: LOAD ERROR {bt['_load_error']} ---")
            continue
        prompt = bt.get("prompt") or {}
        output = bt.get("output") or {}
        nodes = output.get("nodes") or []
        rels = output.get("relationships") or []
        user = prompt.get("user") or ""
        sys_p = prompt.get("system") or ""
        print(f"\n=== {fname} ===")
        print(f"  system_prompt: {len(sys_p):,} chars")
        print(f"  user_prompt:   {len(user):,} chars")
        print(f"  output.nodes:  {len(nodes)}")
        print(f"  output.rels:   {len(rels)}")
        print(f"\n  --- USER PROMPT (first {max_user_chars} chars) ---")
        print(user[:max_user_chars] + ("\n... [truncated]" if len(user) > max_user_chars else ""))
        if nodes:
            print(f"\n  --- OUTPUT NODES (up to {max_output_nodes}) ---")
            for n in nodes[:max_output_nodes]:
                print(f"    {n.get('path','?'):<25s} ids={n.get('ids',{})} props={n.get('properties',{})}")
        else:
            print("\n  --- OUTPUT: empty (this is what the model returned) ---")


# Call it on whichever pass response had failures. Examples:
#   show_failed_batches(resp_id)   # radar_identity
#   show_failed_batches(resp_pr)   # radar_power_rf
#   show_failed_batches(resp_mi)   # missile_identity
# By default below, scan ALL pass responses and show the first one with failures:
_responses = {
    "radar_identity": globals().get("resp_id"),
    "radar_power_rf": globals().get("resp_pr"),
    "radar_antenna": globals().get("resp_an"),
    "radar_timing": globals().get("resp_rt"),
    "radar_modulation": globals().get("resp_rm"),
    "missile_identity": globals().get("resp_mi"),
    "missile_kinematics": globals().get("resp_mk"),
    "missile_guidance": globals().get("resp_mg"),
    "missile_airframe": globals().get("resp_ma"),
    "missile_speed_timing": globals().get("resp_mst"),
    "missile_propulsion": globals().get("resp_mp"),
    "system_links": globals().get("resp_sl"),
}
_found = False
for name, resp in _responses.items():
    if resp is None:
        continue
    diag = (resp or {}).get("diagnostics") or {}
    if diag.get("failed_batch_traces"):
        print(f"=== Pass with failures: {name} ===")
        show_failed_batches(resp)
        _found = True
        break
if not _found:
    print("No pass in this run produced failed_batch_traces. "
          "Either every pass succeeded, or no failure-indicator fired. "
          "Call show_failed_batches(resp_<pass>) directly to force a render.")


No pass in this run produced failed_batch_traces. Either every pass succeeded, or no failure-indicator fired. Call show_failed_batches(resp_<pass>) directly to force a render.


In [29]:
  # ─── Print delta_batch_5 (Run 12's worst-offender chunk) ──────────────────────
  # Reproduces the exact chunking + batching the service uses, no HTTP call.                                                                                                            
                                                                                                                                                                                        
  from docling_core.types.doc import DoclingDocument                                                                                                                                    
  from docling_graph.core.extractors.document_chunker import DocumentChunker                                                                                                            
  from docling_graph.core.extractors.contracts.delta.helpers import (                                                                                                                   
      chunk_batches_by_token_limit,
  )                                                                                                                                                                                     
  from docling_graph.core.extractors.contracts.delta.prompts import (
      format_batch_markdown,                                                                                                                                                            
  )               

  BATCH_INDEX = 5  # delta_batch_5                                                                                                                                                      
   
  docling_doc = DoclingDocument.model_validate(doc)                                                                                                                                     
  chunker = DocumentChunker(
      tokenizer_name="sentence-transformers/all-MiniLM-L6-v2",                                                                                                                          
      chunk_max_tokens=512,                                                                                                                                                             
      merge_peers=True,
  )                                                                                                                                                                                     
  chunks = chunker.chunk_document(docling_doc)
  token_counts = [chunker.tokenizer.count_tokens(c) for c in chunks]                                                                                                                    
  batch_plan = chunk_batches_by_token_limit(chunks, token_counts, max_batch_tokens=1024)                                                                                                
                                                                                                                                                                                        
  if BATCH_INDEX >= len(batch_plan):                                                                                                                                                    
      print(f"batch_index={BATCH_INDEX} out of range (only {len(batch_plan)} batches)")                                                                                                 
  else:                                                                                                                                                                                 
      selected = batch_plan[BATCH_INDEX]                                                                                                                                                
      batch_texts  = [text for _idx, text, _tok in selected]                                                                                                                            
      batch_tokens = sum(tok for _idx, _text, tok in selected)                                                                                                                          
      batch_md     = format_batch_markdown(batch_texts)                                                                                                                                 
                                                                                                                                                                                        
      print(f"========== delta_batch_{BATCH_INDEX} ==========")                                                                                                                         
      print(f"chunks_in_batch = {len(selected)}")
      print(f"total_tokens    = {batch_tokens}")                                                                                                                                        
      print(f"markdown_chars  = {len(batch_md)}")                                                                                                                                       
      print(f"chunk_indices   = {[idx for idx, _, _ in selected]}")                                                                                                                     
      print()                                                                                                                                                                           
                                                                                                                                                                                        
      # Per-chunk view (what the chunker produced)
      for chunk_idx, text, tok in selected:                                                                                                                                             
          print(f"--- chunk[{chunk_idx}]  tokens={tok}  chars={len(text)} ---")                                                                                                         
          print(text)                                                                                                                                                                   
          print()                                                                                                                                                                       
                                                                                                                                                                                        
      # Render exactly as the LLM sees it (rendered markdown the prompt feeds in)                                                                                                       
      print("─" * 60)
      print("RENDERED batch_markdown (what's injected into the user prompt):")                                                                                                          
      print("─" * 60)                                                                                                                                                                   
      print(batch_md)


[DocumentChunker] Initialized with:
  • Tokenizer: sentence-transformers/all-MiniLM-L6-v2
  • Chunk Max Tokens: 512
  • Merge Peers: True

========== delta_batch_5 ==========
chunks_in_batch = 3
total_tokens    = 991
markdown_chars  = 4365
chunk_indices   = [11, 12, 13]

--- chunk[11]  tokens=284  chars=1366 ---
RSNA-75/SNR-75 Fan Song Engagement Radar
The Fan Song is the engagement radar for the S-75/SA-2 family of SAMs. First deployed in strength during the Vietnam conflict, and later used extensively in the Middle East and Africa, the SA-2 was the first Soviet SAM  to  be  used  in anger  and  accounted  for large numbers  of  Western  aircraft until electronic countermeasures were developed. The system was cloned by PLA and still remains widely in use, even though Russia has replaced it with the SA-10/20 system.
The are at least six known variants, one of which is a PLA clone. Details of PLA modifications to the design are  not  public  knowledge.  There  are  sufficient  differences  in  the  PLA  designs  to  regard  these  as  unique derivatives. The antenna configuration of the PLA variants generally follow the Fa

## §19 Field-level temperature A/B/C/D on `radar_power_rf` for "Fan Song"

Runs `radar_power_rf` at T=0.0, 0.1, 0.3, 1.0 against the same document.
Captures the full structured field output for the **Fan Song** entity at each
temperature. Persists each response to disk so partial progress survives
kernel restart, then prints a side-by-side field comparison.

**Cost:** ~17-22 min per call × 4 = ~70-90 min total. Each response is
saved to `/tmp/field_ab_T{temp}.json` immediately on completion.

**Uses existing `doc`** (DoclingDocument loaded earlier via DOCUMENT_SOURCE='real').


In [17]:
# Field-level field-fidelity A/B/C/D across temperature.
# Runs radar_power_rf at 4 temperatures, captures Fan Song's fields each time.
import json, os, time
from pathlib import Path

TEMP_RUNS = [0.0, 0.1, 0.3, 1.0]
TARGET_PASS = "radar_power_rf"
TARGET_ENTITY_PATTERN = "fan song"   # case-insensitive substring match
OUT_DIR = Path("/tmp")
OUT_DIR.mkdir(parents=True, exist_ok=True)

results: dict[float, dict] = {}

for T in TEMP_RUNS:
    out_path = OUT_DIR / f"field_ab_T{T}.json"
    if out_path.exists():
        print(f"[T={T}] cached response found at {out_path}, loading")
        with out_path.open() as f:
            results[T] = json.load(f)
        continue
    print(f"[T={T}] calling /extract-pass pass={TARGET_PASS} ... (this takes ~17-22 min)")
    t0 = time.monotonic()
    _req, resp, elapsed = call_extract_pass(
        TARGET_PASS,
        doc,
        temperature=T,
        llm_batch_token_size=1024,
    )
    print(f"[T={T}] done in {elapsed/60:.1f} min — saving to {out_path}")
    with out_path.open("w") as f:
        json.dump(resp, f, indent=2)
    results[T] = resp

print("\n" + "=" * 78)
print(f"All 4 temperature responses captured. Comparing fields for entities matching")
print(f"  '{TARGET_ENTITY_PATTERN}' (case-insensitive substring) in pass={TARGET_PASS}.")
print("=" * 78)

# Extract Fan Song entities from each response
def extract_target_entities(resp: dict, pattern: str) -> list[dict]:
    p = pattern.lower()
    out = []
    pass_output = resp.get("pass_output") or {}
    for key, items in pass_output.items():
        if not isinstance(items, list):
            continue
        for item in items:
            if not isinstance(item, dict):
                continue
            name = (item.get("system_name") or item.get("name") or "").lower()
            if p in name:
                out.append({"_collection": key, **item})
    return out

matches: dict[float, list[dict]] = {T: extract_target_entities(results[T], TARGET_ENTITY_PATTERN) for T in TEMP_RUNS}

for T in TEMP_RUNS:
    print(f"\n──── T={T} ──── matched {len(matches[T])} entity record(s) for '{TARGET_ENTITY_PATTERN}'")
    for m in matches[T]:
        name = m.get("system_name") or m.get("name") or "<unnamed>"
        print(f"    • {name}")

# Build a flat field-by-field comparison across temperatures.
# Strategy: index matches by their normalized system_name; for each unique name,
# show the field set across the 4 runs side by side.
print("\n" + "=" * 78)
print(f"Field-level diff (per matched entity, per temperature)")
print("=" * 78)

all_names: set[str] = set()
by_temp_by_name: dict[float, dict[str, dict]] = {}
for T in TEMP_RUNS:
    by_temp_by_name[T] = {}
    for m in matches[T]:
        nm = (m.get("system_name") or m.get("name") or "").strip()
        if not nm:
            continue
        # If multiple records share a name (rare), keep the most-populated one.
        prev = by_temp_by_name[T].get(nm)
        if prev is None or len(json.dumps(m)) > len(json.dumps(prev)):
            by_temp_by_name[T][nm] = m
        all_names.add(nm)

for nm in sorted(all_names):
    print(f"\n========== entity: {nm} ==========")
    # Collect the union of all field keys this entity has across temperatures
    field_keys: set[str] = set()
    for T in TEMP_RUNS:
        rec = by_temp_by_name[T].get(nm) or {}
        field_keys.update(k for k in rec.keys() if k not in ("_collection", "system_name", "name"))
    if not field_keys:
        print("  (no fields extracted by any temperature — entity present, fields empty)")
        for T in TEMP_RUNS:
            print(f"  T={T}: {'present' if nm in by_temp_by_name[T] else 'ABSENT'}")
        continue
    # Print a per-field row showing each temperature's value
    width = max(len(k) for k in field_keys) + 2
    header = f"  {'field'.ljust(width)} | {'T=0.0'.ljust(35)} | {'T=0.1'.ljust(35)} | {'T=0.3'.ljust(35)} | {'T=1.0'.ljust(35)}"
    print(header)
    print("  " + "-" * (len(header) - 2))
    for fk in sorted(field_keys):
        cells = []
        for T in TEMP_RUNS:
            rec = by_temp_by_name[T].get(nm) or {}
            v = rec.get(fk, "<missing>")
            if isinstance(v, (dict, list)):
                v = json.dumps(v, separators=(",", ":"))
            cells.append(str(v)[:33])
        row = f"  {fk.ljust(width)} | " + " | ".join(c.ljust(35) for c in cells)
        print(row)

# Quick numeric divergence summary: of the field values where ALL 4 temperatures
# returned a value, count how many cases the values were exactly identical.
print("\n" + "=" * 78)
print("Field-value agreement summary")
print("=" * 78)
all_same = 0
all_diff = 0
partial = 0
missing_at_some = 0
for nm in sorted(all_names):
    field_keys = set()
    for T in TEMP_RUNS:
        rec = by_temp_by_name[T].get(nm) or {}
        field_keys.update(k for k in rec.keys() if k not in ("_collection", "system_name", "name"))
    for fk in field_keys:
        vals = []
        for T in TEMP_RUNS:
            rec = by_temp_by_name[T].get(nm) or {}
            if fk not in rec:
                vals.append("<MISSING>")
            else:
                v = rec.get(fk)
                if isinstance(v, (dict, list)):
                    v = json.dumps(v, sort_keys=True)
                vals.append(str(v))
        if any(v == "<MISSING>" for v in vals):
            missing_at_some += 1
        elif len(set(vals)) == 1:
            all_same += 1
        elif len(set(vals)) == 4:
            all_diff += 1
        else:
            partial += 1
print(f"  Identical across all 4 temperatures:  {all_same}")
print(f"  Two or three values agree, others differ:  {partial}")
print(f"  All 4 differ:  {all_diff}")
print(f"  Missing in at least one temperature:  {missing_at_some}")


[T=0.0] calling /extract-pass pass=radar_power_rf ... (this takes ~17-22 min)
[T=0.0] done in 20.9 min — saving to /tmp/field_ab_T0.0.json
[T=0.1] calling /extract-pass pass=radar_power_rf ... (this takes ~17-22 min)
[T=0.1] done in 21.6 min — saving to /tmp/field_ab_T0.1.json
[T=0.3] calling /extract-pass pass=radar_power_rf ... (this takes ~17-22 min)
[T=0.3] done in 21.2 min — saving to /tmp/field_ab_T0.3.json
[T=1.0] calling /extract-pass pass=radar_power_rf ... (this takes ~17-22 min)
[T=1.0] done in 13.5 min — saving to /tmp/field_ab_T1.0.json

All 4 temperature responses captured. Comparing fields for entities matching
  'fan song' (case-insensitive substring) in pass=radar_power_rf.

──── T=0.0 ──── matched 7 entity record(s) for 'fan song'
    • Fan Song
    • RSNA-75/SNR-75 Fan Song
    • RSNA/SNR-75M Fan Song E
    • SNR-75 PV Cabin / Fan Song
    • SNR-75 Fan Song
    • SNR-75M Fan Song E
    • SNR-75M3 Fan Song

──── T=0.1 ──── matched 8 entity record(s) for 'fan song'
   

## §20 Field-level temperature A/B/C/D — table-heavy missile passes

Sweeps **4 schemas × 3 temperatures = 12 calls** to measure the row-attribution
fix's effect on numeric-field extraction. T=0.0 dropped from prior sweep —
Run 17 showed it was the only temperature that produced wrong values
(2 ✗ on missile_propulsion sustain_mass_kg) while T≥0.1 had zero. All four
schemas have ground-truth data in the SA-2 PDF's variants table at line 251+:

| Schema | Numeric fields | Source |
|---|---|---|
| `missile_kinematics` | min/max_intercept_km, min/max_altitude_km | Max Range, Max Alt rows |
| `missile_airframe` | body_length_m, body_diameter_m, total_mass_kg | Length, Diameter, Weight rows |
| `missile_speed_timing` | max_speed_mps, average_speed_mps | Max speed, Vmax appr tgt rows |
| `missile_propulsion` | booster_mass_kg, sustain_mass_kg, booster_time_sec | 1st/2nd Stage Weight + prose |

**Three optimization goals tracked jointly:**
1. **Entity count** — how many missile variants extracted per (pass, temp)
2. **Speed** — wall time per call, total wall per temperature
3. **Numeric-field extraction** — % populated per field, GT-correctness scorecard

**Cost:** ~22-30 min × 12 = ~4.5-6 hours total. Each response cached to
`/tmp/field_ab_{pass}_T{temp}.json` so partial progress survives.

**Note:** This run measures the **Option A** code change (per-identifier
keyed table-row hints). Prior cached results from Run 17 (row-attribution v1)
have been moved to `/tmp/r17_v1_backup/` for baseline comparison.


In [ ]:
# §20 — Joint sweep: 4 missile schemas × 3 temperatures.
# Goals: entity count, speed, numeric-field extraction (with GT comparison).
import json, time
from pathlib import Path

TEMP_RUNS = [0.1, 0.3, 1.0]
PASSES = [
    "missile_kinematics",
    "missile_airframe",
    "missile_speed_timing",
    "missile_propulsion",
]
OUT_DIR = Path("/tmp")

# Schema field maps — what each pass should populate
SCHEMA_FIELDS = {
    "missile_kinematics":  ["system_name", "min_intercept_km", "max_intercept_km",
                            "min_altitude_km", "max_altitude_km", "max_launch_angle_deg"],
    "missile_airframe":    ["system_name", "body_length_m", "body_diameter_m", "total_mass_kg"],
    "missile_speed_timing":["system_name", "average_speed_mps", "max_speed_mps",
                            "max_flyout_time_sec", "flight_time_sec", "coast_time_sec",
                            "intra_salvo_time_sec", "total_burn_time_sec", "ejector_time_sec"],
    "missile_propulsion":  ["system_name", "ejector_thrust", "ejector_mass_kg",
                            "booster_time_sec", "booster_thrust", "booster_mass_kg",
                            "sustain_time_sec", "sustain_thrust", "sustain_mass_kg"],
}

# Ground truth from the variants table (line 251+ of the SA-2 PDF).
# NATO designations and Fan Song variants share rows; missile column is the
# canonical key for these per-row specs.
GT = {
    "1D":     {"max_intercept_km": 29, "max_altitude_km": 22, "min_altitude_km": 3,    "min_intercept_km": 8,
               "body_length_m": 10.726, "body_diameter_m": 0.654, "total_mass_kg": 2163,
               "booster_mass_kg": 1135, "sustain_mass_kg": 1028,
               "booster_time_sec": 4.0},
    "13D":    {"max_intercept_km": 34, "max_altitude_km": 27, "min_altitude_km": 3,    "min_intercept_km": 8,
               "max_speed_mps": 650,
               "body_length_m": 10.841, "body_diameter_m": 0.654, "total_mass_kg": 2283,
               "booster_mass_kg": 1032, "sustain_mass_kg": 1251,
               "booster_time_sec": 4.0},
    "13DM":   {"max_intercept_km": 43, "max_altitude_km": 30, "min_altitude_km": 1,
               "max_speed_mps": 650,
               "body_length_m": 10.841, "body_diameter_m": 0.654, "total_mass_kg": 2283,
               "booster_mass_kg": 1032, "sustain_mass_kg": 1251},
    "13DA":   {"max_intercept_km": 34, "max_altitude_km": 27, "min_altitude_km": 0.5,
               "max_speed_mps": 650,
               "body_length_m": 10.841, "body_diameter_m": 0.654, "total_mass_kg": 2289,
               "booster_mass_kg": 1032, "sustain_mass_kg": 1257},
    "13DAM":  {"max_intercept_km": 43, "max_altitude_km": 30, "min_altitude_km": 0.3,
               "max_speed_mps": 650,
               "body_length_m": 10.841, "body_diameter_m": 0.654, "total_mass_kg": 2289,
               "booster_mass_kg": 1032, "sustain_mass_kg": 1257},
    "20D":    {"max_intercept_km": 43, "max_altitude_km": 30, "min_altitude_km": 1,
               "max_speed_mps": 885,
               "body_length_m": 10.778, "body_diameter_m": 0.654, "total_mass_kg": 2391,
               "booster_mass_kg": 1011, "sustain_mass_kg": 1380},
    "20DP":   {"max_intercept_km": 45, "max_altitude_km": 30, "min_altitude_km": 1,
               "max_speed_mps": 885,
               "body_length_m": 10.778, "body_diameter_m": 0.654, "total_mass_kg": 2391,
               "booster_mass_kg": 1011, "sustain_mass_kg": 1380},
    "20DSU":  {"max_intercept_km": 56, "max_altitude_km": 30, "min_altitude_km": 0.1,
               "max_speed_mps": 885,
               "body_length_m": 10.778, "body_diameter_m": 0.654, "total_mass_kg": 2397,
               "booster_mass_kg": 1011, "sustain_mass_kg": 1386},
    "5Ya23":  {"max_intercept_km": 76, "max_altitude_km": 30, "min_altitude_km": 0.05,
               "body_length_m": 10.798, "body_diameter_m": 0.654, "total_mass_kg": 2406,
               "booster_mass_kg": 1007, "sustain_mass_kg": 1399},
    "15D":    {"max_intercept_km": 76, "max_altitude_km": 30, "min_altitude_km": 5,
               "body_length_m": 11.200, "body_diameter_m": 0.500, "total_mass_kg": 2450},
}

# ---- Run the 16-call sweep ------------------------------------------------
results: dict[tuple[str, float], dict] = {}
wall_by_cell: dict[tuple[str, float], float] = {}

for pass_name in PASSES:
    print(f"\n{'#' * 78}\n# Pass: {pass_name}\n{'#' * 78}")
    for T in TEMP_RUNS:
        out_path = OUT_DIR / f"field_ab_{pass_name}_T{T}.json"
        if out_path.exists():
            print(f"  [T={T}] cached at {out_path}")
            with out_path.open() as f:
                results[(pass_name, T)] = json.load(f)
            continue
        print(f"  [T={T}] calling /extract-pass ... (~22-30 min)")
        _req, resp, elapsed = call_extract_pass(
            pass_name, doc, temperature=T, llm_batch_token_size=1024,
        )
        wall_by_cell[(pass_name, T)] = elapsed
        print(f"  [T={T}] done in {elapsed/60:.1f} min — saving")
        with out_path.open("w") as f:
            json.dump(resp, f, indent=2)
        results[(pass_name, T)] = resp

# ---- Joint analysis: entity count + speed + numeric-field extraction ------
print("\n\n" + "=" * 100)
print("JOINT METRICS — entity count + numeric-field population per (pass, temperature)")
print("=" * 100)

for pass_name in PASSES:
    print(f"\n──── {pass_name} ────")
    fields = SCHEMA_FIELDS[pass_name]
    numeric_fields = [f for f in fields if f != "system_name"]
    header = f"{'metric':<28} | " + " | ".join(f"T={T}".rjust(15) for T in TEMP_RUNS)
    print(header)
    print("-" * len(header))
    # Row 1: entity count
    row = f"{'entity count':<28} | "
    cells = []
    for T in TEMP_RUNS:
        items = (results[(pass_name, T)].get("pass_output") or {}).get("missile_systems") or []
        cells.append(str(len(items)))
    print(row + " | ".join(c.rjust(15) for c in cells))
    # Row 2: wall time (min)
    row = f"{'wall (min)':<28} | "
    cells = []
    for T in TEMP_RUNS:
        w = wall_by_cell.get((pass_name, T))
        cells.append(f"{w/60:.1f}" if w is not None else "cached")
    print(row + " | ".join(c.rjust(15) for c in cells))
    # Per-field population rates
    for fk in fields:
        cells = []
        for T in TEMP_RUNS:
            items = (results[(pass_name, T)].get("pass_output") or {}).get("missile_systems") or []
            n = len(items)
            pop = sum(1 for e in items if e.get(fk) not in (None, "", [], {}))
            pct = (pop * 100 / n) if n else 0
            cells.append(f"{pop}/{n} ({pct:.0f}%)")
        print(f"{fk:<28} | " + " | ".join(c.rjust(15) for c in cells))

# ---- GT scorecard for missile_kinematics + airframe + speed_timing + propulsion ----
print("\n\n" + "=" * 100)
print("GROUND-TRUTH SCORECARD per (pass, temperature)")
print("   ✓ exact (within tolerance) | ~ close (≤10%) | ✗ wrong (>10%) | — null")
print("=" * 100)

def grade(actual, expected, abs_tol=0.5, rel_tol=0.10):
    if actual is None: return "—"
    if expected is None: return " "
    try:
        a, e = float(actual), float(expected)
        if abs(a - e) < abs_tol: return "✓"
        if abs(a - e) / max(abs(e), 1) < rel_tol: return "~"
        return "✗"
    except (TypeError, ValueError):
        return "?"

GT_FIELDS_PER_PASS = {
    "missile_kinematics": ["max_intercept_km", "max_altitude_km", "min_altitude_km", "min_intercept_km"],
    "missile_airframe": ["body_length_m", "body_diameter_m", "total_mass_kg"],
    "missile_speed_timing": ["max_speed_mps"],
    "missile_propulsion": ["booster_mass_kg", "sustain_mass_kg", "booster_time_sec"],
}
ABS_TOL = {  # field-specific absolute tolerance for ✓
    "body_length_m": 0.05,
    "body_diameter_m": 0.005,
    "total_mass_kg": 5.0,
    "max_speed_mps": 5.0,
    "booster_mass_kg": 5.0,
    "sustain_mass_kg": 5.0,
    "booster_time_sec": 0.5,
}

for pass_name in PASSES:
    fields_to_check = GT_FIELDS_PER_PASS.get(pass_name, [])
    if not fields_to_check:
        continue
    print(f"\n──── {pass_name} ────")
    print(f"{'missile':<8} | {'field':<22} | " + " | ".join(f"T={T}".rjust(20) for T in TEMP_RUNS))
    print("-" * 110)
    for missile_name, gt in GT.items():
        for fk in fields_to_check:
            if fk not in gt:
                continue
            row = f"{missile_name:<8} | {fk:<22} | "
            cells = []
            for T in TEMP_RUNS:
                items = (results[(pass_name, T)].get("pass_output") or {}).get("missile_systems") or []
                e = next((x for x in items if x.get("system_name") == missile_name), None)
                if e is None:
                    cells.append("ABSENT")
                    continue
                v = e.get(fk)
                tol = ABS_TOL.get(fk, 0.5)
                g = grade(v, gt[fk], abs_tol=tol)
                cells.append(f"{v} {g} (gt={gt[fk]})")
            print(row + " | ".join(c.rjust(20)[:20] for c in cells))

# ---- Aggregate scorecard per (pass, temperature) -------------------------
print("\n\n" + "=" * 100)
print("AGGREGATE SCORECARD per (pass, temperature) — GT field accuracy across 10 missile variants")
print("=" * 100)
print(f"{'pass':<22} | {'T':>5} | {'present':>8} | {'✓ exact':>8} | {'~ close':>8} | {'✗ wrong':>8} | {'— null':>8}")
print("-" * 100)
for pass_name in PASSES:
    fields_to_check = GT_FIELDS_PER_PASS.get(pass_name, [])
    if not fields_to_check:
        continue
    for T in TEMP_RUNS:
        items = (results[(pass_name, T)].get("pass_output") or {}).get("missile_systems") or []
        by_name = {x.get("system_name"): x for x in items}
        present = exact = close = wrong = null = 0
        for missile_name, gt in GT.items():
            for fk in fields_to_check:
                if fk not in gt:
                    continue
                e = by_name.get(missile_name)
                if e is None:
                    null += 1
                    continue
                present += 1 if e else 0
                v = e.get(fk)
                tol = ABS_TOL.get(fk, 0.5)
                g = grade(v, gt[fk], abs_tol=tol)
                if   g == "✓": exact += 1
                elif g == "~": close += 1
                elif g == "✗": wrong += 1
                else: null += 1
        print(f"{pass_name:<22} | {T:>5} | {present:>8} | {exact:>8} | {close:>8} | {wrong:>8} | {null:>8}")

print("\nKey: present = entity row found in output (regardless of whether GT field populated)")
print("     null    = entity absent OR field null when GT exists")



##############################################################################
# Pass: missile_kinematics
##############################################################################
  [T=0.1] calling /extract-pass ... (~22-30 min)


In [10]:
  import urllib.request, json
  api = "http://api:8000"
  # pick the SA-2 doc id (the notebook resolves it via REAL_DOCUMENT_NAME)
  resp = urllib.request.urlopen(f"{api}/v1/documents/{DOC_ID}/docling")
  d = json.load(resp)
  print("tables:", len(d.get("tables", [])))
  print("body refs:", [c for c in d["body"]["children"] if "tables" in str(c)][:5])
  print("first table cells:", d["tables"][0]["data"]["table_cells"][:6] if d.get("tables") else "none")

NameError: name 'DOC_ID' is not defined